---
# INTERVIEW PREPARATION & KEY TAKEAWAYS

## Critical Concepts Every Data Engineer Must Know

### Performance Optimization Hierarchy
1. **Query tuning first** (rewrite logic)
2. **Indexing second** (create strategic indexes)
3. **Partitioning third** (split large tables)
4. **Denormalization last** (if all else fails)

### Common Interview Questions & Gotchas

**Q: How would you optimize a slow query?**
```
Answer: (In order)
1. Run EXPLAIN to see execution plan
2. Check if needed indexes exist
3. Verify index is being used (key column = not NULL)
4. Consider composite indexes (column order matters)
5. Check for full table scans (type=ALL = bad)
6. Rewrite query if index can't help
7. Consider materialized views or denormalization
```

**Q: What's the difference between TRUNCATE and DELETE?**
```
Key points to mention:
- TRUNCATE is DDL, DELETE is DML
- TRUNCATE cannot be rolled back in MyISAM
- TRUNCATE resets AUTO_INCREMENT, DELETE doesn't
- TRUNCATE doesn't fire triggers
- TRUNCATE is 10-100x faster
```

**Q: How do you handle NULL values?**
```
Must know:
- NULL != NULL (both return NULL, not TRUE)
- NULL propagates in math (NULL + 5 = NULL)
- Use IS NULL / IS NOT NULL (not = NULL)
- COALESCE() to provide defaults
- NULL in NOT IN returns no rows (subtle!)
- Aggregations ignore NULLs (COUNT vs COUNT(*))
```

**Q: Explain STAR vs Snowflake schema?**
```
Key differences:
- STAR: Denormalized (fast queries, data redundancy)
- Snowflake: Normalized (storage efficient, more joins)
- STAR has fewer joins (2-3 vs 4-5)
- Snowflake easier to maintain (single source of truth)
- Choice depends on query vs storage priority
```

**Q: What's partition pruning?**
```
Must explain:
- Query planner eliminates partitions not needed
- Example: Range partition by date, query WHERE date < '2024-01-01'
- Only scans p2023 partition (not p2024, p2025)
- Can 10-100x query performance on large tables
- Requires partition key in WHERE clause to work
```

**Q: ACID vs BASE consistency models?**
```
ACID (Relational DBs like MySQL):
- Atomicity: All-or-nothing
- Consistency: Valid state maintained
- Isolation: No interference
- Durability: Survives crashes

BASE (NoSQL):
- Basically Available
- Soft state (eventually consistent)
- Eventually consistent
- Trade strong consistency for availability/scale
```

## Best Practices for Data Engineers

### Do's ✅
- Always use EXPLAIN before production query
- Create indexes on columns in WHERE, JOIN, ORDER BY
- Use LIMIT during development (avoid full table scans)
- Test transactions with concurrent load
- Document schema design decisions
- Use PRIMARY KEY and UNIQUE constraints
- Monitor slow query log regularly
- Archive old data (TRUNCATE old partitions)

### Don'ts ❌
- Don't use SELECT * in production (explicit columns)
- Don't create too many indexes (INSERT/UPDATE penalty)
- Don't use implicit type conversions in WHERE
- Don't mix MyISAM and InnoDB in same transaction
- Don't grant SUPER privilege to users
- Don't use % in GRANT statements
- Don't rely on LAST_INSERT_ID() for sequences
- Don't assume EXPLAIN rows are accurate

## Summary of All Sections Covered

| Section | Key Insight |
|---------|-------------|
| Data Movement | LOAD DATA fastest, TRUNCATE non-recoverable in MyISAM |
| Data Integrity | CAST kills indexes, AUTO_INCREMENT has gaps, INSERT ON DUPLICATE KEY for upserts |
| Schema Design | STAR=fast queries, Snowflake=low storage |
| Indexes | B-Tree for ranges, column order critical, covering indexes avoid table access |
| Partitioning | Range/List for pruning, Hash for distribution |
| Query Optimization | EXPLAIN type=const best, avoid full table scans |
| Transformations | PIVOT requires CASE in MySQL, GROUPING SETS/CUBE for hierarchies |
| Set Operations | UNION slow (dedup), UNION ALL fast, EXISTS better than IN |
| Recursive CTEs | Need depth limit, exponential performance degradation |
| NULL Handling | NULL != NULL, propagates in math, IS NULL required |
| Engines | InnoDB ACID+rows locks, MyISAM fast but no transactions |
| Transactions | REPEATABLE READ prevents non-repeatable but allows phantom reads |
| Locks | Row locks in InnoDB, deadlock detection automatic |
| Security | Principle of least privilege, avoid % wildcards |
| Connectivity | Default bind-address=localhost (not accessible remotely) |
| Nested Data | JSON not indexed by default, self-joins for hierarchies |

## Resources for Further Learning
- MySQL 8.0 Documentation
- Use Case: "SELECT * FROM logs WHERE ERROR HANDLING" (real examples matter!)
- Practice on LeetCode SQL problems (top 50 for data engineers)

---
# SECTION 16: NESTED DATA HANDLING - JSON, HIERARCHIES & ADVANCED PATTERNS

### 16.1: JSON Data Types

**Storing JSON:**
```sql
-- Create table with JSON column
CREATE TABLE employee_profiles (
    emp_id INT PRIMARY KEY,
    emp_name VARCHAR(100),
    profile JSON  -- Store semi-structured data
);

-- Insert JSON data
INSERT INTO employee_profiles (emp_id, emp_name, profile) VALUES
(1, 'Alice', '{
    "phone": "555-1234",
    "address": {
        "city": "SF",
        "zip": "94105"
    },
    "skills": ["SQL", "Python", "Java"]
}');
```

**Query JSON:**
```sql
-- Extract JSON value
SELECT 
    emp_id,
    JSON_EXTRACT(profile, '$.phone') as phone,
    JSON_EXTRACT(profile, '$.address.city') as city
FROM employee_profiles
WHERE emp_id = 1;

-- Array iteration (explode)
SELECT 
    emp_id,
    JSON_UNQUOTE(JSON_EXTRACT(profile, '$.skills[*]')) as skill
FROM employee_profiles
WHERE JSON_CONTAINS(profile, '["SQL"]', '$.skills');

-- Update JSON
UPDATE employee_profiles
SET profile = JSON_SET(profile, '$.phone', '555-9999')
WHERE emp_id = 1;
```

### 16.2: Self-Joins for Hierarchies

**Org Chart with Self-Join:**
```sql
SELECT 
    e.emp_id,
    e.emp_name,
    m.emp_name as manager_name,
    e.salary,
    m.salary as manager_salary
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.manager_id, e.emp_name;

-- Result shows each employee with their manager
-- Left join because top manager has no manager
```

**Find Peer Group (same manager):**
```sql
SELECT 
    e1.emp_id,
    e1.emp_name,
    GROUP_CONCAT(e2.emp_name) as peers
FROM employees e1
JOIN employees e2 ON e1.manager_id = e2.manager_id 
                   AND e1.emp_id != e2.emp_id
GROUP BY e1.emp_id, e1.emp_name;
```

### 16.3: Complex Aggregation Patterns

**Cumulative Totals (Running Sum):**
```sql
SELECT 
    emp_id,
    sale_date,
    sale_amount,
    SUM(sale_amount) OVER (
        PARTITION BY emp_id 
        ORDER BY sale_date
    ) as cumulative_total
FROM sales
ORDER BY emp_id, sale_date;
```

**Row Ranking:**
```sql
SELECT 
    emp_id,
    sale_amount,
    ROW_NUMBER() OVER (ORDER BY sale_amount DESC) as rank,
    DENSE_RANK() OVER (ORDER BY sale_amount DESC) as dense_rank,
    PERCENT_RANK() OVER (ORDER BY sale_amount DESC) as percent_rank
FROM sales;
```

**⚠️ Gotchas:**
- **JSON not indexed by default** (slow queries)
- **Semi-structured data = loose schema** (validation needed in app)
- **Self-joins can cartesian product** if conditions wrong
- **Window functions require MySQL 8.0+** (earlier versions don't support)

### 14.2: Connectivity & Remote Connections

**Configure Remote Access:**
```sql
-- Check current bind-address
SHOW VARIABLES LIKE 'bind_address';
-- Result: localhost or 127.0.0.1 (local only!)

-- Edit my.cnf to allow remote connections
[mysqld]
bind-address = 0.0.0.0  -- Listen on all interfaces
# or
bind-address = 192.168.1.100  -- Specific IP

-- Restart MySQL
sudo systemctl restart mysql
```

**Connection from Remote Host:**
```bash
# Connect from another machine
mysql -h 192.168.1.100 -u data_engineer -p company_db

# Connection string in Python/Java
# jdbc:mysql://192.168.1.100:3306/company_db?user=data_engineer&password=xxx
```

**Change Default Port:**
```sql
-- Edit my.cnf
[mysqld]
port = 3307  -- Change from default 3306

-- Restart MySQL
sudo systemctl restart mysql

-- Connect to new port
mysql -h localhost -u root -p -P 3307
```

**SSH Tunneling (Secure Remote Access):**
```bash
# Create SSH tunnel through jumphost
ssh -L 3306:localhost:3306 user@jumphost.example.com

# Then connect locally (tunneled through SSH)
mysql -h localhost -u data_engineer -p
```

**⚠️ Gotchas:**
- **Default bind-address=127.0.0.1** (localhost only!)
- **Port 3306 is obvious to attackers** (security through obscurity? No)
- **Firewall must allow port** (UFW, iptables rules)
  ```bash
  sudo ufw allow 3307/tcp from 192.168.1.0/24
  ```
- **Host names in GRANT** - resolve correctly!
  ```sql
  GRANT SELECT ON company_db.* TO 'user'@'host.example.com';
  -- Host must resolve via DNS or be in /etc/hosts
  ```
- **% wildcard** - dangerous (accepts any origin!)

---
# SECTION 14: SECURITY & CONNECTIVITY - PRIVILEGES & ACCESS CONTROL

### 14.1: User Management & GRANT/REVOKE

**Create Users:**
```sql
-- Create user with password
CREATE USER 'data_engineer'@'localhost' IDENTIFIED BY 'secure_password';

-- Create user from any host
CREATE USER 'analyst'@'%' IDENTIFIED BY 'password123';
```

**GRANT Privileges - Database Level:**
```sql
-- SELECT only (read-only analyst)
GRANT SELECT ON company_db.* TO 'analyst'@'localhost';

-- Full database access (data engineer)
GRANT SELECT, INSERT, UPDATE, DELETE ON company_db.* TO 'data_engineer'@'localhost';

-- All privileges (admin)
GRANT ALL PRIVILEGES ON company_db.* TO 'admin_user'@'localhost';
```

**GRANT Privileges - Table Level:**
```sql
-- Specific tables
GRANT SELECT ON company_db.employees TO 'hr_reader'@'localhost';
GRANT SELECT, UPDATE ON company_db.employees TO 'hr_manager'@'localhost';
```

**GRANT Privileges - Column Level:**
```sql
-- Grant access to specific columns (salary sensitive)
GRANT SELECT (emp_id, emp_name, dept_id) ON company_db.employees TO 'manager'@'localhost';
-- Manager can see ID, name, dept but NOT salary!
```

**REVOKE Privileges:**
```sql
-- Remove specific privilege
REVOKE INSERT ON company_db.* FROM 'analyst'@'localhost';

-- Remove all privileges
REVOKE ALL PRIVILEGES ON company_db.* FROM 'data_engineer'@'localhost';

-- Revoke privilege from user entirely
DROP USER 'analyst'@'localhost';
```

**⚠️ Critical Gotchas:**
- **Wildcard users (%)** - accessible from any host (security risk!)
- **SUPER privilege** - can bypass all restrictions (dangerous!)
- **Column-level grants** - complex to manage (consider views instead)
- **WITH GRANT OPTION** - user can grant to others (use sparingly)
  ```sql
  GRANT SELECT ON company_db.* TO 'admin'@'localhost' WITH GRANT OPTION;
  -- Admin can now GRANT SELECT to other users
  ```
- **Principle of least privilege** - grant minimum needed only

---
# SECTION 13: RELIABILITY - LOCKS, SAVEPOINTS & MVCC

### 13.1: Database Locks

**Row-Level Locks (InnoDB):**
```sql
-- Shared lock (S lock) - multiple transactions can read
SELECT * FROM employees WHERE emp_id = 1 LOCK IN SHARE MODE;
-- Other transactions can also read but not modify

-- Exclusive lock (X lock) - only one transaction can access
SELECT * FROM employees WHERE emp_id = 1 FOR UPDATE;
-- Other transactions must wait

-- Update acquires exclusive lock automatically
BEGIN;
UPDATE employees SET salary = 100000 WHERE emp_id = 1;
-- Locks row exclusively until COMMIT
```

**Deadlock Scenario:**
```sql
-- Connection 1              | Connection 2
BEGIN;                       | BEGIN;
UPDATE emp SET sal=100       |  UPDATE emp SET sal=200
  WHERE emp_id = 1;          |    WHERE emp_id = 2;
                             | UPDATE emp SET sal=300
                             |   WHERE emp_id = 1;  ← WAITING
UPDATE emp SET sal=400       | ← DEADLOCK DETECTED!
  WHERE emp_id = 2;          |
-- Error: Deadlock found     | Rolls back automatically
```

**Deadlock Resolution:**
```sql
-- Check lock waits
SELECT * FROM INFORMATION_SCHEMA.PROCESSLIST 
WHERE COMMAND = 'Sleep' AND TIME > 30;

-- Kill stuck transaction
KILL 123;  -- Connection ID 123

-- InnoDB automatically chooses victim (smallest transaction)
```

### 13.2: SAVEPOINT - Partial Rollback

**Syntax:**
```sql
BEGIN;
UPDATE employees SET salary = 100000;
SAVEPOINT sp1;  -- Mark point 1

UPDATE employees SET salary = 200000;
SAVEPOINT sp2;  -- Mark point 2

DELETE FROM sales;  -- Oops, mistake!
ROLLBACK TO sp2;  -- Undo DELETE, keep other updates
COMMIT;  -- Permanent
```

**Nested Transactions Example:**
```sql
BEGIN;
INSERT INTO employees (emp_name, dept_id) VALUES ('Alice', 1);
SAVEPOINT sp_emp;

INSERT INTO sales (emp_id, amount) VALUES (11, 1000);
SAVEPOINT sp_sales;

-- Error occurs in next transaction
INSERT INTO employees (emp_id, ...) -- Error: duplicate key
ROLLBACK TO sp_sales;  -- Undo INSERT INTO sales
-- But Alice still inserted! Continue with valid insert
INSERT INTO sales (emp_id, amount) VALUES (11, 2000);

COMMIT;  -- All changes permanent
```

### 13.3: MVCC (Multi-Version Concurrency Control)

**Concept:** InnoDB maintains multiple versions of data for non-locking reads.

```sql
-- Transaction 1 (reads)       | Transaction 2 (writes)
BEGIN;                         |
SELECT emp_id FROM employees;  |
(reads version 100)            |
                               | BEGIN;
                               | UPDATE employees SET salary = 100000
                               |   WHERE emp_id = 1;
                               | COMMIT;
                               | (version 101 created)
SELECT emp_id FROM employees;  |
(reads SAME version 100!)      |
COMMIT;                        | ← Sees consistent snapshot

-- No locks needed! Transaction 1 sees consistent state throughout
```

**⚠️ Gotchas:**
- **Long transactions hold version chains** (prevents garbage collection)
- **Undo logs grow large** (impacts storage)
- **Phantom reads still possible** in REPEATABLE READ (not truly serializable)

---
# SECTION 12: RELIABILITY - TRANSACTIONS & ACID PROPERTIES

### 12.1: ACID Properties

**Atomicity:** All-or-nothing execution
```sql
BEGIN;
UPDATE employees SET salary = salary + 1000 WHERE dept_id = 1;
UPDATE employees SET salary = salary - 1000 WHERE dept_id = 2;
-- If either fails, BOTH operations roll back (not partial)
COMMIT;  -- Both succeed or both fail
```

**Consistency:** Data validity maintained
```sql
-- All foreign key constraints, unique constraints checked
BEGIN;
INSERT INTO employees (emp_id, emp_name, dept_id) VALUES (11, 'New', 999);
-- Fails if dept_id 999 doesn't exist (FK constraint violation)
-- Transaction rolls back automatically
```

**Isolation:** Concurrent transactions don't interfere
```sql
-- Transaction 1          | Transaction 2
BEGIN;                   | BEGIN;
UPDATE salary...         | 
COMMIT;                  | SELECT salary (sees new value!)
                         | COMMIT;
```

**Durability:** Committed data persists
```sql
BEGIN;
DELETE FROM sales WHERE emp_id = 2;
COMMIT;  -- Data permanently deleted (survives server crash)
```

### 12.2: Transaction Control

```sql
-- Basic transaction
BEGIN;  -- or START TRANSACTION
UPDATE employees SET salary = 100000 WHERE emp_id = 1;
COMMIT;  -- Make permanent

-- Rollback transaction
BEGIN;
UPDATE employees SET salary = 200000 WHERE emp_id = 2;
ROLLBACK;  -- Undo changes (salary not updated)

-- Autocommit behavior
SET AUTOCOMMIT = 0;  -- Disable autocommit
UPDATE employees SET salary = 150000;  -- Not permanent yet
COMMIT;  -- Now permanent

SET AUTOCOMMIT = 1;  -- Enable autocommit (default)
UPDATE employees SET salary = 160000;  -- Permanent immediately
```

### 12.3: Isolation Levels (Prevent Anomalies)

**READ UNCOMMITTED** (lowest isolation)
```sql
SET TRANSACTION ISOLATION LEVEL READ UNCOMMITTED;

-- Allows "dirty reads" (read uncommitted data)
Transaction 1: UPDATE salary... (not committed)
Transaction 2: SELECT salary... (sees UNCOMMITTED value!) ❌ Dirty read
```

**READ COMMITTED** (most common)
```sql
SET TRANSACTION ISOLATION LEVEL READ COMMITTED;

-- Prevents dirty reads but allows non-repeatable reads
Transaction 1: SELECT salary (first: 100000)
Transaction 2:         UPDATE salary = 150000; COMMIT;
Transaction 1: SELECT salary (again: 150000) ❌ Non-repeatable read
```

**REPEATABLE READ** (MySQL default for InnoDB)
```sql
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;

-- Prevents dirty & non-repeatable reads but allows phantom reads
Transaction 1: SELECT * FROM employees WHERE dept_id = 1; (5 rows)
Transaction 2:         INSERT employees (dept_id) VALUES (1); COMMIT;
Transaction 1: SELECT * FROM employees WHERE dept_id = 1; (6 rows!) ❌ Phantom read
```

**SERIALIZABLE** (highest isolation - slowest)
```sql
SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;

-- Transactions run sequentially (no concurrency)
-- Slowest but prevents all anomalies
```

### 11.2: information_schema - Query Metadata

**Query Table Structures:**
```sql
-- Find all VARCHAR columns > 255 characters
SELECT TABLE_NAME, COLUMN_NAME, COLUMN_TYPE, COLUMN_KEY
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'your_db'
  AND COLUMN_TYPE LIKE 'varchar%'
  AND NUMERIC_SCALE > 255;

-- Find all tables with no primary key
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'your_db'
  AND TABLE_TYPE = 'BASE TABLE'
  AND TABLE_NAME NOT IN (
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
    WHERE TABLE_SCHEMA = 'your_db'
      AND CONSTRAINT_NAME = 'PRIMARY'
  );

-- Check index fragmentation
SELECT TABLE_NAME, INDEX_NAME, SEQ_IN_INDEX, COLUMN_NAME, CARDINALITY
FROM INFORMATION_SCHEMA.STATISTICS
WHERE TABLE_SCHEMA = 'your_db'
  AND INDEX_NAME != 'PRIMARY'
ORDER BY TABLE_NAME, INDEX_NAME, SEQ_IN_INDEX;

-- Find key lengths
SELECT TABLE_NAME, COLUMN_NAME, COLUMN_TYPE, CHARACTER_SET_NAME, COLLATION_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'your_db'
  AND DATA_TYPE IN ('varchar', 'char', 'text');
```

**Table Types and Row Formats:**
```sql
-- Check row format (static vs dynamic)
SELECT TABLE_NAME, ROW_FORMAT, DATA_LENGTH, INDEX_LENGTH
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'your_db'
  AND ENGINE = 'InnoDB';

-- Row formats:
-- COMPACT/REDUNDANT: Old (good for small tables)
-- DYNAMIC: Modern (good for LOB data)
-- COMPRESSED: Space efficient (slower)
```

---
# SECTION 11: ENGINE INTERNALS - MyISAM vs InnoDB

### 11.1: MyISAM vs InnoDB Comparison

```sql
-- Create MyISAM table (fast, no transactions, full table locks)
CREATE TABLE sales_myisam (
    sale_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_id INT,
    amount DECIMAL(10,2),
    sale_date DATE,
    INDEX idx_emp (emp_id)
) ENGINE=MyISAM;

-- Create InnoDB table (ACID, row-level locks, transactions)
CREATE TABLE sales_innodb (
    sale_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_id INT,
    amount DECIMAL(10,2),
    sale_date DATE,
    INDEX idx_emp (emp_id)
) ENGINE=InnoDB;

-- Check table engine
SELECT TABLE_NAME, ENGINE FROM INFORMATION_SCHEMA.TABLES 
WHERE TABLE_SCHEMA = 'your_db';
```

| Feature | MyISAM | InnoDB |
|---------|--------|--------|
| Transactions | ❌ NO | ✅ YES (ACID) |
| Locking | Table locks (slow) | Row-level locks (fast) |
| Speed | ⚡ Very Fast | ⚡ Fast (slower than MyISAM) |
| Crash Recovery | ❌ Corrupts easily | ✅ Good (MVCC) |
| Foreign Keys | ❌ NO | ✅ YES |
| Rollback | ❌ NO | ✅ YES |
| Binary Log | ❌ Not recommended | ✅ Reliable |
| Buffer Pool | ❌ Key cache only | ✅ Buffer pool (bigger) |

**⚠️ Gotchas:**
- **MyISAM not recommended** (deprecated, only for special cases like FULLTEXT indexes)
- **Cannot mix engines in transactions** (each isolated)
- **Migration from MyISAM to InnoDB:**
  ```sql
  ALTER TABLE sales_myisam ENGINE=InnoDB;
  -- Takes time, locks table, may corrupt if interrupted
  ```
- **Foreign key constraints only in InnoDB** (MyISAM silently ignores them)
- **MyISAM ibdata1 file grows and never shrinks** (storage issue)

---
# SECTION 10: NULL HANDLING & ADVANCED FUNCTIONS

### 10.1: COALESCE, IFNULL, NULLIF Functions

**Concept:** Handle NULL values in calculations and comparisons.

```sql
-- COALESCE: Return first non-NULL value
SELECT 
    emp_id,
    COALESCE(manager_id, -1) as manager_id,  -- -1 if NULL
    COALESCE(emp_name, 'Unknown') as emp_name
FROM employees;

-- IFNULL: Return alternative if NULL (MySQL specific)
SELECT 
    emp_id,
    IFNULL(manager_id, 0) as manager_id,  -- 0 if NULL
    IFNULL(salary, 0) as salary
FROM employees;

-- NULLIF: Return NULL if two values match
SELECT 
    emp_id,
    NULLIF(salary, 0) as salary  -- NULL if salary = 0, else salary
FROM employees;

-- Avoid division by zero
SELECT 
    emp_id,
    SUM(sale_amount) / NULLIF(COUNT(*), 0) as avg_sale
FROM sales
GROUP BY emp_id;
-- Avoids "Divided by zero error"
```

**⚠️ Critical Gotchas:**
- **NULL propagates in calculations:**
  - `NULL + 5 = NULL`
  - `NULL * 0 = NULL` (not 0!)
  - `NULL > 5` returns NULL (not FALSE)
- **NULL != NULL is TRUE** (in WHERE clause, both return NULL)
  ```sql
  SELECT * FROM employees WHERE manager_id != 1;
  -- Excludes employees with manager_id = NULL!
  -- Use: WHERE manager_id != 1 OR manager_id IS NULL
  ```
- **NULL in GROUP BY creates separate group** (can be surprising)
- **NULL in indexes:** Most DBs allow indexing NULL (treated as group)
- **Three-valued logic:** TRUE, FALSE, NULL (not two!)

### 10.2: NULL in Aggregations

```sql
-- COUNT ignores NULLs
SELECT COUNT(manager_id) FROM employees;  -- Counts non-NULL only

-- COUNT(*) includes NULLs
SELECT COUNT(*) FROM employees;  -- All rows

-- SUM ignores NULLs
SELECT SUM(salary) FROM employees;  -- Only non-NULL values

-- AVG ignores NULLs (careful!)
SELECT AVG(salary) FROM employees;  -- Only non-NULL, divides by non-NULL count
```

---
# SECTION 9: COMPLEX LOGIC - RECURSIVE CTEs & SUBQUERIES

### 9.1: Recursive CTEs - Hierarchical Data

**Concept:** Use `WITH RECURSIVE` to traverse trees (org charts, hierarchies).

```sql
-- Org chart: Find all subordinates of Alice (emp_id=1)
WITH RECURSIVE org_hierarchy AS (
    -- Base case: Alice (manager_id IS NULL)
    SELECT emp_id, emp_name, manager_id, 1 as depth
    FROM employees
    WHERE emp_id = 1
    
    UNION ALL
    
    -- Recursive case: Find direct reports
    SELECT e.emp_id, e.emp_name, e.manager_id, oh.depth + 1
    FROM employees e
    INNER JOIN org_hierarchy oh ON e.manager_id = oh.emp_id
    WHERE oh.depth < 10  -- Prevent infinite loops
)
SELECT * FROM org_hierarchy
ORDER BY depth, emp_name;

-- Result:
-- emp_id | emp_name | manager_id | depth
-- 1      | Alice    | NULL       | 1
-- 2      | Bob      | 1          | 2
-- 3      | Carol    | 1          | 2
-- 8      | Henry    | 1          | 2
```

**⚠️ Critical Gotchas:**
- **Must have depth limit** (WHERE depth < 10) or UNION DISTINCT, else infinite loop
- **Performance degrades quickly** (exponential with depth)
- **MySQL 5.7+ only** (earlier versions don't support)
- **UNION DISTINCT forces dedup** (slower than UNION ALL)
- **Memory usage high** (entire tree in memory)

### 9.2: Subqueries - ANY, ALL, IN, EXISTS

**Syntax:**
```sql
-- IN (equivalent to multiple ORs)
SELECT * FROM employees 
WHERE dept_id IN (1, 2, 3);
-- Same as: WHERE dept_id = 1 OR dept_id = 2 OR dept_id = 3

-- ANY (comparison with multiple values)
SELECT * FROM employees 
WHERE salary > ANY (SELECT salary FROM employees WHERE dept_id = 2);
-- Returns employees with salary > ANY dept 2 salary

-- ALL (comparison with all values)
SELECT * FROM employees 
WHERE salary > ALL (SELECT salary FROM employees WHERE dept_id = 2);
-- Returns employees with salary > ALL dept 2 salaries

-- EXISTS (check if subquery returns rows)
SELECT * FROM employees e
WHERE EXISTS (
    SELECT 1 FROM sales s WHERE s.emp_id = e.emp_id AND s.sale_amount > 1000
);
-- Returns employees with at least one sale > $1000
```

**Performance Comparison:**
```
EXISTS > IN > ANY/ALL
 ✅ Fast              ❌ Slow
```

**⚠️ Gotchas:**
- **NOT IN with NULL returns nothing!**
  ```sql
  SELECT * FROM employees WHERE emp_id NOT IN (1, 2, NULL);
  -- Returns 0 rows (NULL makes entire comparison unknown!)
  -- Use: NOT IN (SELECT COALESCE(emp_id, -1) FROM...)
  ```
- **Correlated subqueries run per row** (slow on large tables)
- **EXISTS more efficient** than IN for large subqueries

---
# SECTION 8: SET OPERATIONS - UNION, INTERSECT, EXCEPT

### 8.1: UNION vs UNION ALL

**Concept:** Combine results from multiple queries.

```sql
-- UNION (removes duplicates - slower)
SELECT emp_name FROM employees WHERE dept_id = 1
UNION
SELECT emp_name FROM employees WHERE salary > 100000;

-- Result: [Alice, Bob, Carol, Henry]
-- (If Alice appears in both result sets, listed once)

-- UNION ALL (keeps duplicates - faster)
SELECT emp_name FROM employees WHERE dept_id = 1
UNION ALL
SELECT emp_name FROM employees WHERE salary > 100000;

-- Result: [Alice, Bob, Carol, Henry, Alice, Henry]
-- (Alice and Henry appear twice)
```

**⚠️ Gotchas:**
- **UNION is slower** (requires sorting/dedup)
- **UNION ALL is faster** (no dedup)
- **Column count must match** (error if different)
- **Data types must be compatible** (implicit conversion)

### 8.2: INTERSECT (Common Rows)

**Concept:** Return only rows that appear in BOTH queries.

```sql
-- Find employees in Engineering dept AND with salary > $80K
SELECT emp_name FROM employees WHERE dept_id = 1
INTERSECT
SELECT emp_name FROM employees WHERE salary > 80000;

-- Result: [Bob, Carol, Henry]
-- (Only these are in BOTH result sets)
```

### 8.3: EXCEPT / MINUS (Differences)

**Concept:** Return rows in first query but NOT in second query.

```sql
-- Find employees in Engineering dept NOT earning > $100K
SELECT emp_name FROM employees WHERE dept_id = 1
EXCEPT
SELECT emp_name FROM employees WHERE salary > 100000;

-- Result: [Bob, Carol]
-- (In Dept 1 but salary <= $100K)
```

**Performance Comparison:**
```
UNION ALL > INTERSECT > EXCEPT > UNION
 ✅ Fast                        ❌ Slow
```

---
# SECTION 7: ADVANCED TRANSFORMATIONS - PIVOT/UNPIVOT & GROUPING

### 7.1: PIVOT (Rows to Columns) - MySQL Implementation

**Concept:** Convert rows into columns (reshape data for reporting).

**MySQL Workaround (no native PIVOT):**
```sql
-- Sample data: Sales by employee per month
SELECT emp_id, emp_name, MONTH(sale_date) as month, SUM(sale_amount) as sales
FROM sales s
JOIN employees e ON s.emp_id = e.emp_id
GROUP BY emp_id, emp_name, MONTH(sale_date);

-- Result:
-- emp_id | emp_name | month | sales
-- 2      | Bob      | 1     | 1275
-- 2      | Bob      | 2     | 400
-- 5      | Eve      | 1     | 1500

-- PIVOT using CASE/SUM
SELECT 
    emp_id,
    emp_name,
    SUM(CASE WHEN MONTH(sale_date) = 1 THEN sale_amount ELSE 0 END) as jan_sales,
    SUM(CASE WHEN MONTH(sale_date) = 2 THEN sale_amount ELSE 0 END) as feb_sales,
    SUM(CASE WHEN MONTH(sale_date) = 3 THEN sale_amount ELSE 0 END) as mar_sales
FROM sales s
JOIN employees e ON s.emp_id = e.emp_id
GROUP BY emp_id, emp_name;

-- Result:
-- emp_id | emp_name | jan_sales | feb_sales | mar_sales
-- 2      | Bob      | 1275      | 400       | 0
-- 5      | Eve      | 1500      | 0         | 0
```

**⚠️ Gotchas:**
- **MySQL lacks native PIVOT** (use CASE/SUM workaround)
- **NULL handling:** Missing values become 0 or NULL depending on logic
- **Performance:** Heavy GROUP BY on pre-pivoted data

### 7.2: GROUPING SETS, CUBE, ROLLUP

**Syntax:**
```sql
-- GROUPING SETS: Multiple grouping levels in one query
SELECT 
    dept_id,
    emp_id,
    SUM(sale_amount) as total_sales
FROM sales s
JOIN employees e ON s.emp_id = e.emp_id
GROUP BY GROUPING SETS (
    (dept_id),           -- Sales by department
    (emp_id),            -- Sales by employee
    ()                   -- Grand total
);

-- Result:
-- dept_id | emp_id | total_sales
-- 1       | NULL   | 3000        (Dept 1 total)
-- 2       | NULL   | 1500        (Dept 2 total)
-- NULL    | 2      | 1675        (Employee 2 total)
-- NULL    | 5      | 1500        (Employee 5 total)
-- NULL    | NULL   | 4675        (Grand total)

-- ROLLUP: Hierarchical aggregation
SELECT 
    YEAR(sale_date) as year,
    MONTH(sale_date) as month,
    SUM(sale_amount) as sales
FROM sales
GROUP BY ROLLUP(YEAR(sale_date), MONTH(sale_date));

-- Result shows: Year total → Month total → Grand total

-- CUBE: All possible combinations
SELECT 
    dept_id,
    YEAR(sale_date) as year,
    SUM(sale_amount) as sales
FROM sales s
JOIN employees e ON s.emp_id = e.emp_id
GROUP BY CUBE(dept_id, YEAR(sale_date));

-- Result includes: (dept, year), (dept, NULL), (NULL, year), (NULL, NULL)
```

---
# SECTION 6: PERFORMANCE & SCALING - PARTITIONING

### 6.1: Range Partitioning - Date-Based

**Concept:** Split data into ranges based on column values (usually date).

```sql
CREATE TABLE sales (
    sale_id INT,
    emp_id INT,
    sale_amount DECIMAL(10,2),
    sale_date DATE,
    PRIMARY KEY (sale_id, sale_date)  -- Partition key must be in PK
) PARTITION BY RANGE (YEAR(sale_date)) (
    PARTITION p2022 VALUES LESS THAN (2023),
    PARTITION p2023 VALUES LESS THAN (2024),
    PARTITION p2024 VALUES LESS THAN (2025),
    PARTITION p_future VALUES LESS THAN MAXVALUE
);
```

**Benefits:**
✅ **Query pruning:** `WHERE sale_date < '2023-01-01'` only scans p2022
✅ **Easy maintenance:** Drop old partitions instead of DELETE
✅ **Archive strategy:** Move old partitions to cold storage

**⚠️ Gotchas:**
- **Partition key must be in PRIMARY KEY** (or unique index)
- **EXPLAIN won't show pruning** (run with WHERE clause)
- **Maintenance overhead:** Must add new partitions manually or via stored procedure

### 6.2: List Partitioning - Categorical

**Concept:** Split data into discrete value lists.

```sql
CREATE TABLE employees (
    emp_id INT PRIMARY KEY,
    emp_name VARCHAR(100),
    dept_id INT,
    salary DECIMAL(10,2)
) PARTITION BY LIST (dept_id) (
    PARTITION p_eng VALUES IN (1, 2, 3),     -- Engineering depts
    PARTITION p_sales VALUES IN (2),          -- Sales dept
    PARTITION p_other VALUES IN (4, 5, 6)     -- Other depts
);
```

### 6.3: Hash Partitioning - Distribution

**Concept:** Distribute rows based on hash of column (even distribution).

```sql
CREATE TABLE sales (
    sale_id INT PRIMARY KEY,
    sale_amount DECIMAL(10,2),
    emp_id INT
) PARTITION BY HASH (emp_id) PARTITIONS 8;
-- Creates 8 partitions distributed evenly
```

**Benefits:**
✅ **Even distribution** (no skewed partitions)
✅ **Good for scaling** (add more partitions as data grows)

**⚠️ Gotchas:**
- **Cannot do partition pruning** (must scan all partitions)
- **Slower than Range/List** (requires hash computation)

### 5.4: EXPLAIN - Understanding Query Execution Plans

**Syntax:**
```sql
EXPLAIN SELECT * FROM employees WHERE dept_id = 1 AND salary > 80000;

-- Extended information (JSON format)
EXPLAIN FORMAT=JSON SELECT * FROM employees WHERE dept_id = 1;

-- Show rewritten query
EXPLAIN EXTENDED SELECT * FROM employees WHERE dept_id = 1;
SHOW WARNINGS;
```

**Key Columns in EXPLAIN Output:**

| Column | Meaning |
|--------|---------|
| id | SELECT identifier (subquery order) |
| select_type | SIMPLE, PRIMARY, SUBQUERY, UNION, etc. |
| table | Which table |
| **type** | **Access type: ALL, index, range, ref, const** |
| possible_keys | Indexes MySQL could use |
| **key** | **Index actually used** |
| key_len | Length of index key (helps diagnose partial usage) |
| rows | **Estimated # of rows examined** |
| Extra | Using where, Using index, Using temporary, etc. |

**Type Performance Ranking:**
```
const > eq_ref > ref > range > index > ALL
✅ FAST                              ❌ SLOW
```

**Example Analysis:**
```sql
EXPLAIN SELECT * FROM employees WHERE emp_id = 5;
-- type: const (best - single row by primary key)

EXPLAIN SELECT * FROM employees WHERE dept_id = 1;
-- type: ref (good - multiple rows by non-unique index)

EXPLAIN SELECT * FROM employees WHERE salary > 80000;
-- type: range (good - range of values)

EXPLAIN SELECT * FROM employees;
-- type: ALL (bad - full table scan)
```

**⚠️ Gotchas:**
- **EXPLAIN is estimated!** Actual execution may differ
- **Statistics can be outdated** (run ANALYZE TABLE if needed)
- **type = ALL means full table scan** (always bad for large tables)
- **rows can be wildly inaccurate** on large tables (sample-based estimation)
- **Key_len mismatch** indicates partial index use (wrong column order?)
- **Extra = Using index** is good (index-only scan)
- **Extra = Using temporary** is bad (creates temp table during grouping)

### 5.2: Composite Indexes - Multiple Columns

**Concept:** Index on multiple columns for queries involving all of them.

**Syntax:**
```sql
-- Composite index on (dept_id, hire_date)
CREATE INDEX idx_dept_hire ON employees(dept_id, hire_date);

-- Query using both columns (✅ index used)
SELECT * FROM employees 
WHERE dept_id = 1 AND hire_date > '2020-01-01';
-- ✅ Index used efficiently

-- Query using only dept_id (✅ index used - leftmost prefix)
SELECT * FROM employees WHERE dept_id = 1;
-- ✅ Index used (leftmost column)

-- Query using only hire_date (❌ index NOT used)
SELECT * FROM employees WHERE hire_date > '2020-01-01';
-- ❌ Full table scan (not leftmost column)
```

**⚠️ Critical Gotchas:**
- **Column order matters!** Put most selective column first
- **Leftmost prefix rule:** Can only use leftmost portion of composite index
- **High cardinality first:** Index low-cardinality columns AFTER high-cardinality
  - Example: For `(dept_id, salary)` - salary first is better (many unique values)
- **Skip scanning:** MySQL 5.6+ can skip non-leading columns (but inefficient)

### 5.3: Covering Indexes - Index-Only Scans

**Concept:** Index contains all columns needed for a query, avoiding table access.

```sql
-- Create covering index
CREATE INDEX idx_covering ON employees(dept_id, salary, hire_date);

-- Query accesses ONLY index (no table lookup needed!)
SELECT salary, hire_date FROM employees 
WHERE dept_id = 1;
-- ✅ Index-only scan (very fast!)

-- Query needs emp_id (not in index)
SELECT emp_id, salary FROM employees 
WHERE dept_id = 1;
-- ❌ Must access table (emp_id not covered)
```

**⚠️ Gotchas:**
- **Large indexes use more memory** and impact RAM
- **Over-indexing kills write performance** (INSERT/UPDATE maintain all indexes)
- **Index fragmentation increases** (more data = more splits)
- **Diminishing returns** (covering index for 10 columns = excessive overhead)

---
# SECTION 5: PERFORMANCE & SCALING - INDEXES

## The 'Why'
Indexes speed up data retrieval at the cost of:
- **Slower writes** (INSERT, UPDATE, DELETE must maintain index)
- **Storage overhead** (indexes consume disk space)
- **Memory usage** (hot indexes cached in buffer pool)

### 5.1: B-Tree vs Hash Indexes

**B-Tree Indexes** (default in MySQL/InnoDB)
- Data structure: Balanced tree
- Best for: Range queries, sorting, wildcards
- Example: `WHERE salary > 80000` or `WHERE emp_name LIKE 'A%'`

```sql
-- B-Tree index (default type)
CREATE INDEX idx_salary ON employees(salary);

-- Range query uses index efficiently
SELECT * FROM employees WHERE salary BETWEEN 80000 AND 120000;
-- ✅ Index range scan (fast!)

-- Prefix search works with B-Tree
SELECT * FROM employees WHERE emp_name LIKE 'A%';
-- ✅ Index used (starts with 'A')
```

**Hash Indexes** (Memory engine only - rarely used)
- Data structure: Hash table
- Best for: Exact matches only
- Example: `WHERE emp_id = 5` (not ranges!)

```sql
-- Hash index (only for Memory engine, not InnoDB)
CREATE TABLE hash_example (
    id INT PRIMARY KEY,
    value VARCHAR(100),
    KEY hash_idx (value) USING HASH
) ENGINE=MEMORY;

-- Exact match is fastest
SELECT * FROM hash_example WHERE value = 'Alice';
-- ✅✅ Hash lookup (very fast)

-- Range query on hash index = SLOW
SELECT * FROM hash_example WHERE value > 'M';
-- ❌ Full table scan (hash can't do ranges)
```

| Feature | B-Tree | Hash |
|---------|--------|------|
| Exact match | ✅ Fast | ✅✅ Faster |
| Range query | ✅ Fast | ❌ Full scan |
| Sorting | ✅ Sorted order | ❌ Unsorted |
| Prefix search | ✅ `LIKE 'A%'` | ❌ No |
| NULL handling | ✅ Good | ⚠️ Can crash |
| Engines | InnoDB, MyISAM | Memory only |

### 4.3: Materialized Views - Pre-computed Results

**Concept:** A view that stores actual data on disk (not computed on-the-fly). Trades freshness for performance.

**Syntax (Standard SQL):**
```sql
-- NOT supported in MySQL natively, use table + refresh logic
CREATE MATERIALIZED VIEW sales_summary_mv AS
SELECT 
    emp_id,
    emp_name,
    dept_id,
    YEAR(sale_date) as sale_year,
    MONTH(sale_date) as sale_month,
    SUM(sale_amount) as total_sales,
    COUNT(*) as num_transactions,
    AVG(sale_amount) as avg_sale
FROM sales
JOIN employees ON sales.emp_id = employees.emp_id
GROUP BY emp_id, emp_name, dept_id, YEAR(sale_date), MONTH(sale_date);
```

**MySQL Implementation (using table + trigger):**
```sql
-- Create materialized view as table
CREATE TABLE sales_summary AS
SELECT 
    emp_id,
    emp_name,
    dept_id,
    YEAR(sale_date) as sale_year,
    MONTH(sale_date) as sale_month,
    SUM(sale_amount) as total_sales,
    COUNT(*) as num_transactions,
    AVG(sale_amount) as avg_sale
FROM sales
JOIN employees ON sales.emp_id = employees.emp_id
GROUP BY emp_id, emp_name, dept_id, YEAR(sale_date), MONTH(sale_date);

-- Add index for fast queries
CREATE INDEX idx_emp_date ON sales_summary(emp_id, sale_year, sale_month);
```

**Refresh Strategies:**

**1. Manual Refresh (scheduled job)**
```sql
-- Daily refresh at midnight
TRUNCATE TABLE sales_summary;
INSERT INTO sales_summary SELECT ... FROM sales ...;
```

**2. On Demand Refresh**
```sql
-- Create view that queries source (not truly materialized)
CREATE VIEW sales_summary_view AS
SELECT ... FROM sales ...;

-- Query materialized view or refresh when needed
SELECT * FROM sales_summary_view WHERE emp_id = 2;
```

**3. Incremental Refresh (only new/changed data)**
```sql
-- Only refresh today's sales (faster!)
INSERT INTO sales_summary SELECT ... FROM sales 
WHERE sale_date >= DATE(NOW());

-- Or use REPLACE to update changed rows
REPLACE INTO sales_summary SELECT ... FROM sales 
WHERE sale_date >= DATE(NOW());
```

**⚠️ Gotchas:**
- **Stale data** (always behind source tables)
- **Refresh overhead** (full recalc = expensive for large tables)
- **Storage cost** (duplicate data, 2x the space)
- **No automatic refresh in MySQL** (must use stored procedures + triggers)
- **Query optimizer won't automatically use** if you forget to query summary table
- **Incremental refresh complexity** (tracking what changed)

### 4.2: Snowflake Schema - Normalized for Storage

**Concept:** Fact table surrounded by normalized (non-denormalized) dimension tables.

**Schema Definition:**
```sql
-- Fact table (minimal denormalization)
CREATE TABLE fact_sales (
    sale_id INT PRIMARY KEY,
    emp_id INT,
    product_id INT,
    dept_id INT,
    date_id INT,
    sale_amount DECIMAL(10,2),
    quantity INT
);

-- Normalized employee dimension (NO dept_name)
CREATE TABLE dim_employee (
    emp_id INT PRIMARY KEY,
    emp_name VARCHAR(100),
    dept_id INT,  -- Foreign key to dim_department
    salary DECIMAL(10,2),
    hire_date DATE
);

-- Separate department dimension
CREATE TABLE dim_department (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(100),
    location VARCHAR(100),
    budget DECIMAL(12,2)
);

-- Product dimension
CREATE TABLE dim_product (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(150),
    category VARCHAR(50),
    price DECIMAL(10,2)
);
```

**Query Example (more joins than STAR):**
```sql
-- SNOWFLAKE: Requires additional join to dim_department
SELECT 
    SUM(fs.sale_amount) as total_sales,
    dd.dept_name
FROM fact_sales fs
JOIN dim_employee de ON fs.emp_id = de.emp_id
JOIN dim_department dd ON de.dept_id = dd.dept_id  -- ← Additional join
GROUP BY dd.dept_name;
```

**Benefits:**
✅ **Lower storage** (no redundancy)
✅ **Easier updates** (single source of truth for dept_name)
✅ **Better for OLTP integration** (native normalized form)

**Drawbacks:**
❌ **More joins** (slower queries) - 2+ joins typical
❌ **Complex for analysts** (more relationships to learn)
❌ **Join overhead** (network I/O, memory usage)

### Comparison Summary

| Aspect | STAR | Snowflake |
|--------|------|-----------|
| Joins | 1-2 | 2+ |
| Query Speed | ⚡⚡ Fast | ⚡ Slower |
| Storage | 📦 More | 📦📦 Less |
| Updates | ❌ Anomalies | ✅ Safe |
| Complexity | 🔵 Simple | 🟢 Complex |

---
# SECTION 4: SCHEMA DESIGN - STAR VS SNOWFLAKE

## The 'Why'
Schema design determines:
- **Query performance** (denormalization → fewer joins)
- **Storage efficiency** (normalization → less redundancy)
- **Update costs** (anomalies in denormalized schemas)
- **Reporting speed** (materialized views trade freshness for speed)

### 4.1: STAR Schema - Denormalized for Analytics

**Concept:** One denormalized FACT table surrounded by DIMENSION tables.

**Visual Structure:**
```
         DIM_EMPLOYEE
       /              \
      /                \
   DIM_DATE ---- FACT_SALES ---- DIM_PRODUCT
      \                /
       \              /
        DIM_DEPARTMENT
```

**Schema Definition:**
```sql
-- Fact Table (contains dimension IDs and measures, DENORMALIZED)
CREATE TABLE fact_sales (
    sale_id INT PRIMARY KEY,
    emp_id INT,
    product_id INT,
    dept_id INT,
    date_id INT,
    sale_amount DECIMAL(10,2),
    quantity INT,
    INDEX idx_emp (emp_id),
    INDEX idx_product (product_id),
    INDEX idx_dept (dept_id),
    INDEX idx_date (date_id)
);

-- Dimension Table (DENORMALIZED - all attributes in one table)
CREATE TABLE dim_employee (
    emp_id INT PRIMARY KEY,
    emp_name VARCHAR(100),
    dept_id INT,
    dept_name VARCHAR(100),  -- ← DENORMALIZED! (also in dim_department)
    salary DECIMAL(10,2),
    hire_date DATE
);

CREATE TABLE dim_product (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(150),
    category VARCHAR(50),
    price DECIMAL(10,2)
);

CREATE TABLE dim_department (
    dept_id INT PRIMARY KEY,
    dept_name VARCHAR(100),
    location VARCHAR(100)
);

CREATE TABLE dim_date (
    date_id INT PRIMARY KEY,  -- YYYYMMDD format: 20240115
    calendar_date DATE,
    year INT, month INT, day INT,
    quarter INT, day_of_week INT
);
```

**Benefits:**
✅ **Few joins** (simple queries) - usually 1-2 joins per query
✅ **Fast aggregations** (minimal GROUP BY operations)
✅ **Easy for business analysts** to understand

**Drawbacks:**
❌ **Data redundancy** (dept_name stored in both employee & department dims)
❌ **Update anomalies** (if dept name changes, must update everywhere)
❌ **Storage bloat** (10-50% more space)
❌ **Complex ETL** (must denormalize during load)

### 3.3: INSERT ... ON DUPLICATE KEY UPDATE (Upsert Pattern)

**Concept:** If PRIMARY KEY or UNIQUE constraint exists, UPDATE the row. Otherwise, INSERT.

**Syntax:**
```sql
INSERT INTO employees (emp_id, emp_name, salary, dept_id, hire_date)
VALUES (2, 'Bob Smith Updated', 100000, 1, '2019-03-20')
ON DUPLICATE KEY UPDATE
    emp_name = 'Bob Smith Updated',
    salary = 100000;
```

**More Complex Example with Increment:**
```sql
-- Increment salary by 10% if employee exists, or insert new
INSERT INTO employees (emp_id, emp_name, salary, dept_id, hire_date)
VALUES (5, 'Eve Davis', 82500, 2, '2020-06-15')
ON DUPLICATE KEY UPDATE
    salary = salary * 1.1,
    emp_name = NEW.emp_name;
```

**Batch Upsert Example:**
```sql
-- Upsert multiple rows efficiently
INSERT INTO employees (emp_id, emp_name, salary, dept_id, hire_date) VALUES
  (2, 'Bob Updated', 98000, 1, '2019-03-20'),
  (11, 'New Employee', 88000, 1, '2024-03-01')
ON DUPLICATE KEY UPDATE
  emp_name = NEW.emp_name,
  salary = NEW.salary;
```

**Returned Rows:**
- If INSERT: affected_rows = 1
- If UPDATE: affected_rows = 2 (MySQL counts it as 2!)

**⚠️ Gotchas:**
- **VALUES() function deprecated** (MySQL 8.0.20+). Use `NEW.column` instead:
  ```sql
  INSERT INTO ... ON DUPLICATE KEY UPDATE
    column = NEW.column;  -- Modern syntax
  ```
- **Triggers fire for both INSERT and UPDATE** (expensive on 1M row batches)
- **Only works with PRIMARY KEY and UNIQUE constraints**
- **If multiple unique keys exist, first violation wins** (may not be the one you want)
- **Cannot use in multi-table INSERT**
- **Binary logging treats as UPDATE** (replication implications)

### 3.2: AUTO_INCREMENT Sequences - Unique ID Generation

**Concept:** Automatically generates unique sequential numbers for each new row.

**Syntax:**
```sql
CREATE TABLE employees (
    emp_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_name VARCHAR(100),
    salary DECIMAL(10,2),
    INDEX idx_emp_id (emp_id)
);

-- Insert without specifying emp_id
INSERT INTO employees (emp_name, salary) VALUES ('New Employee', 75000);
-- emp_id automatically becomes 11 (next available)

-- Retrieve last inserted ID
SELECT LAST_INSERT_ID();  -- Returns 11
```

**Multi-Server Setup (replication safe):**
```sql
-- Server 1: odd IDs only (1, 3, 5, 7...)
SET @@auto_increment_increment = 2;
SET @@auto_increment_offset = 1;
SET @@auto_increment_start = 1;

-- Server 2: even IDs only (2, 4, 6, 8...)
SET @@auto_increment_increment = 2;
SET @@auto_increment_offset = 2;
SET @@auto_increment_start = 2;
```

**⚠️ Gotchas:**
- **Gaps in sequences:** TRUNCATE resets to 1, but DELETE doesn't
  - `DELETE FROM employees; -- emp_id counter stays at 11`
  - `TRUNCATE TABLE employees; -- emp_id counter resets to 1`
- **LAST_INSERT_ID() returns 0** if last INSERT didn't use AUTO_INCREMENT
- **No gap-free sequences** (use trigger with LOCK if you need contiguous IDs)
- **Not suitable for secure ticket systems** (can predict IDs)
- **In replication:** Can cause ID conflicts on master-master setups
- **ROLLBACK doesn't return ID** (counter advances even if transaction rolls back)
- **Offset persists across restarts** (stored in table metadata)

---
# SECTION 3: DATA INTEGRITY - TYPE CASTING, SEQUENCES, & UPSERT

## The 'Why'
Data integrity ensures:
- **Correct data types** for calculations and comparisons
- **Unique identifiers** via sequences/auto-increment
- **No duplicate or conflicting records** via upsert logic

### 3.1: CAST & CONVERT - Explicit Type Conversion

**Syntax:**
```sql
-- CAST (ANSI standard, portable across DBs)
SELECT CAST(salary AS CHAR) FROM employees;
SELECT CAST('2024-01-15' AS DATE) FROM dual;
SELECT CAST('12345' AS SIGNED INTEGER) FROM dual;

-- CONVERT (MySQL specific)
SELECT CONVERT(salary, CHAR) FROM employees;
SELECT CONVERT('2024-01-15', DATE) FROM dual;
SELECT CONVERT(12345 USING UTF8MB4) FROM dual;
```

**Common Conversions:**
```sql
-- String to Number
SELECT CAST('12345' AS UNSIGNED);  -- 12345
SELECT CAST('12.99' AS DECIMAL(5,2));  -- 12.99

-- Number to String (right-padded in CHAR)
SELECT CAST(12345 AS CHAR(10));  -- '12345     '

-- String to Date
SELECT CAST('2024-01-15' AS DATE);  -- 2024-01-15
SELECT STR_TO_DATE('15/01/2024', '%d/%m/%Y');  -- 2024-01-15

-- Date to String formatting
SELECT DATE_FORMAT(hire_date, '%d/%m/%Y') FROM employees;  -- '15/01/2024'
```

**⚠️ Gotchas:**
- **Implicit casting in WHERE KILLS INDEXES!**
  - Bad: `WHERE CAST(emp_id AS VARCHAR) = '2'` → full table scan
  - Good: `WHERE emp_id = 2` → index used
- **Data loss in narrowing conversions:**
  - `CAST(123.99 AS INT)` → 123 (truncated, not rounded)
  - `CAST('12345ABCD' AS INT)` → 12345 (stops at non-numeric)
- **Leading zeros lost in numeric conversions:**
  - `CAST('00123' AS INT)` → 123 (then back to string = '123')

### 2.4: TRUNCATE - Fast Table Clear (DDL Operation)

**Concept:** Remove all rows from a table instantly. DDL operation, not DML.

**Syntax:**
```sql
TRUNCATE TABLE sales;  -- Instant, cannot be rolled back in MyISAM
```

**TRUNCATE vs DELETE Comparison:**

| Feature | TRUNCATE | DELETE |
|---------|----------|--------|
| Type | DDL (Data Definition) | DML (Data Manipulation) |
| Speed | Instant (no row scan) | Slow (scans all rows) |
| Space Deallocation | Deallocates | Keeps allocated |
| WHERE clause | ❌ NO | ✅ YES |
| Triggers | ❌ NO | ✅ YES |
| Rollback | ❌ NO (MyISAM) ✅ YES (InnoDB) | ✅ YES |
| Binary Logging | 👁️ Special handling | ✅ Standard log |
| AUTO_INCREMENT | Reset to 1 | Preserved |

**Examples:**
```sql
-- Fast clear (but cannot rollback in MyISAM!)
TRUNCATE TABLE sales;

-- Delete specific rows (slower but precise)
DELETE FROM sales WHERE emp_id = 5;

-- Delete with condition (slower but can rollback)
DELETE FROM sales WHERE sale_date < '2024-01-01';
COMMIT;  -- Make permanent
```

**⚠️ Critical Gotchas:**
- **Cannot rollback TRUNCATE in MyISAM!** Even if wrapped in transaction
- **InnoDB can rollback TRUNCATE** (since MySQL 5.6)
- **Resets AUTO_INCREMENT counter** to 1 (can break sequences if you rely on gaps)
- **Doesn't fire DELETE triggers** (can miss cleanup logic)
- **Cannot use with WHERE clause** (must use DELETE if need partial clear)
- **Fails silently on foreign key constraints** (depends on engine)
- **Slower on MyISAM with many tables** (must rebuild table structure)

### 2.3: REPLACE - Destructive Upsert

**Concept:** If row exists (matching unique key), DELETE it then INSERT new one. If not exists, just INSERT.

**Syntax:**
```sql
REPLACE INTO employees (emp_id, emp_name, dept_id, salary, hire_date)
VALUES (2, 'Bob Smith Updated', 1, 100000, '2019-03-20');
```

**Behavior:**
- Deletes old row with emp_id=2
- Inserts new row with updated values
- Generates DELETE + INSERT log entries (slower than INSERT IGNORE)

**⚠️ Critical Gotchas:**
- **Can lose data!** If row has other columns not in REPLACE, they get NULLed
  - Example: If your REPLACE doesn't include hire_date, hire_date becomes NULL!
- **AUTO_INCREMENT may generate new IDs**, creating gaps and breaking sequences
- **Triggers on DELETE and INSERT both fire** (very expensive on large batches)
- **Foreign key constraints may be violated!** Child records pointing to old row lost
- **Slower than INSERT ... ON DUPLICATE KEY UPDATE**

**Better Alternative:** Use `INSERT ... ON DUPLICATE KEY UPDATE` (see Section 3)

### 2.2: INSERT IGNORE - Graceful Duplicate Handling

**Concept:** Insert rows but skip if they violate unique constraints or primary keys.

**Syntax:**
```sql
INSERT IGNORE INTO employees (emp_id, emp_name, dept_id, salary, hire_date)
VALUES (1, 'Alice Johnson', 1, 120000, '2018-01-15');
```

**Behavior:**
- If no constraint violation: inserts normally (affected_rows = 1)
- If constraint violated: ignores the row silently (affected_rows = 0)
- Other data quality issues (type mismatch) cause NULL insertion or truncation

**Example Scenario:**
```sql
INSERT IGNORE INTO employees (emp_id, emp_name, dept_id, salary, hire_date)
VALUES 
  (2, 'Bob Smith', 1, 95000, '2019-03-20'),  -- Already exists, will be skipped
  (11, 'New Employee', 1, 88000, '2024-03-01'); -- New, will insert
-- Result: 1 row inserted, 1 row skipped
```

**⚠️ Gotchas:**
- Silently skips rows (audit nightmare!). Use error logs or separate validation instead
- Returns misleading affected_rows count
- Does NOT generate errors that you can catch in application code
- MySQL 8.0.20+: Use ON CONFLICT clause for more control
- Partial inserts succeed (inconsistent batch behavior)

## COMPREHENSIVE CONTENT ADDITIONS

All remaining sections have been organized logically. The notebook continues with:

**2.2: INSERT IGNORE** - Skip duplicates gracefully
**2.3: REPLACE** - Destructive upsert with gotchas
**2.4: TRUNCATE** - Fast table clear (DDL vs DML)

**SECTION 3: DATA INTEGRITY**
- Type Casting: CAST, CONVERT, implicit conversions
- AUTO_INCREMENT sequences: gaps, multi-server setup  
- INSERT ... ON DUPLICATE KEY UPDATE: upsert logic

**SECTION 4: SCHEMA DESIGN**
- STAR Schema: denormalized fact + dimension tables
- Snowflake Schema: normalized dimensions
- Materialized Views: Simple, Complex, On Demand refresh

**SECTION 5: PERFORMANCE & INDEXES**
- B-Tree vs Hash indexes
- Composite indexes (column order matters)
- Covering indexes (index-only scans)
- EXPLAIN output analysis

**SECTION 6: PARTITIONING**
- Range partitioning (date-based)
- List partitioning (categorical)
- Hash partitioning (distribution)
- Partition pruning & maintenance

**SECTION 7: QUERY OPTIMIZATION**
- EXPLAIN FORMAT=JSON deep dive
- Index selection strategies
- Query rewriting techniques
- Join optimization

**SECTION 8: ADVANCED TRANSFORMATIONS**
- PIVOT/UNPIVOT (MySQL CASE workarounds)
- GROUPING SETS, CUBE, ROLLUP
- Multi-level aggregations
- Hierarchical reporting

**SECTION 9: SET OPERATIONS**
- UNION vs UNION ALL
- INTERSECT (common rows)
- EXCEPT/MINUS (differences)
- Data reconciliation patterns

**SECTION 10: RECURSIVE CTEs & COMPLEX LOGIC**
- Recursive CTE syntax: WITH RECURSIVE
- Org chart traversal (hierarchies)
- Bill of Materials (BOM) tree structures
- Cycle detection & depth limits
- Subqueries: ANY, ALL, EXISTS, IN
- Correlated vs uncorrelated subqueries
- Performance implications

**SECTION 11: NULL HANDLING**
- COALESCE, IFNULL, NULLIF functions
- NULL in WHERE, JOIN, aggregation contexts
- Three-valued logic (TRUE, FALSE, NULL)
- NULL in indexes & performance

**SECTION 12: ENGINE INTERNALS**
- MyISAM: fast, no transactions, table locks
- InnoDB: ACID compliant, row-level locks
- Storage comparison & migration
- information_schema queries

**SECTION 13: TRANSACTIONS & ACID**
- ACID properties: Atomicity, Consistency, Isolation, Durability
- BEGIN, COMMIT, ROLLBACK
- Isolation levels: READ UNCOMMITTED → SERIALIZABLE
- Dirty reads, non-repeatable reads, phantom reads

**SECTION 14: LOCKS & SAVEPOINTS**
- Row vs Table locks
- Shared (S) vs Exclusive (X) locks
- Deadlock scenarios & resolution
- SAVEPOINT for partial rollbacks
- MVCC (Multi-Version Concurrency Control)

**SECTION 15: SECURITY - GRANT/REVOKE**
- User privileges: SELECT, INSERT, UPDATE, DELETE
- Database, table, column-level grants
- User creation & password management
- Principle of least privilege
- SUPER privilege security

**SECTION 16: CONNECTIVITY & CONFIGURATION**
- bind-address for remote connections
- Changing default port 3306
- SSH tunneling & SSL/TLS setup
- Firewall rules & access control

**SECTION 17: NESTED DATA**
- JSON data types: JSON_EXTRACT, JSON_SET
- Self-joins for hierarchical data
- Complex aggregation patterns
- Semi-structured data handling

# Mastering SQL for Data Engineering
## A Comprehensive Guide to Interview Preparation

**Course Level:** Advanced | **Target Audience:** Data Engineers, Analytics Engineers, Database Administrators

---

### Notebook Overview
This notebook covers **20 core SQL topics** essential for data engineering roles:
1. Data Movement - Import/Export & Bulk Operations
2. Data Integrity - Type Casting, Sequences, & Upsert Logic
3. Schema Design - STAR vs Snowflake & Materialized Views
4. Performance & Scaling - Indexes
5. Performance & Scaling - Partitioning
6. Query Optimization - EXPLAIN Plans
7. Advanced Transformations - Pivot, Unpivot, & Grouping
8. Set Operations - Union, Intersect, Except
9. Complex Logic - Recursive CTEs & Subqueries
10. NULL Handling & Advanced Functions
11. Engine Internals - MyISAM vs InnoDB
12. Transactions & ACID Compliance
13. Database Locks, Savepoints, & MVCC
14. Security - Privileges, GRANT/REVOKE
15. Remote Connections & Port Configuration
16. Nested Data Handling

---
# SECTION 1: SETUP & MOCK DATASETS

## Create Mock Database for All Examples

### Objective
Establish a realistic mock database environment with:
- **employees** - employee records with salary, department, hire date
- **departments** - department master data
- **sales** - transactional sales data
- **products** - product master data with pricing

These tables represent a typical OLTP database in a mid-sized organization.

### SQL DDL: Create Tables

```sql
-- Create departments table
CREATE TABLE departments (
    dept_id INT PRIMARY KEY AUTO_INCREMENT,
    dept_name VARCHAR(100) NOT NULL UNIQUE,
    location VARCHAR(100),
    budget DECIMAL(12, 2),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create employees table with hierarchical relationship
CREATE TABLE employees (
    emp_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_name VARCHAR(100) NOT NULL,
    dept_id INT NOT NULL,
    salary DECIMAL(10, 2) NOT NULL,
    hire_date DATE NOT NULL,
    manager_id INT,
    FOREIGN KEY (dept_id) REFERENCES departments(dept_id),
    FOREIGN KEY (manager_id) REFERENCES employees(emp_id),
    INDEX idx_dept (dept_id),
    INDEX idx_hire_date (hire_date),
    INDEX idx_manager (manager_id)
);

-- Create products table
CREATE TABLE products (
    product_id INT PRIMARY KEY AUTO_INCREMENT,
    product_name VARCHAR(150) NOT NULL,
    category VARCHAR(50),
    price DECIMAL(10, 2) NOT NULL,
    stock_quantity INT DEFAULT 0,
    created_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Create sales table (transactional fact table)
CREATE TABLE sales (
    sale_id INT PRIMARY KEY AUTO_INCREMENT,
    emp_id INT NOT NULL,
    product_id INT NOT NULL,
    sale_amount DECIMAL(10, 2) NOT NULL,
    sale_date DATE NOT NULL,
    quantity INT NOT NULL DEFAULT 1,
    FOREIGN KEY (emp_id) REFERENCES employees(emp_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    INDEX idx_emp_id (emp_id),
    INDEX idx_sale_date (sale_date),
    INDEX idx_product (product_id)
);
```

### SQL DML: Insert Sample Data

```sql
-- Insert departments
INSERT INTO departments (dept_name, location, budget) VALUES
('Engineering', 'San Francisco', 500000),
('Sales', 'New York', 300000),
('Marketing', 'Boston', 200000),
('Operations', 'Chicago', 250000),
('HR', 'San Francisco', 100000);

-- Insert employees (note: Alice is manager of Bob and Carol)
INSERT INTO employees (emp_name, dept_id, salary, hire_date, manager_id) VALUES
('Alice Johnson', 1, 120000, '2018-01-15', NULL),
('Bob Smith', 1, 95000, '2019-03-20', 1),
('Carol White', 1, 100000, '2019-06-10', 1),
('David Brown', 2, 85000, '2020-01-05', NULL),
('Eve Davis', 2, 75000, '2020-06-15', 4),
('Frank Miller', 3, 80000, '2019-09-01', NULL),
('Grace Lee', 4, 90000, '2018-11-20', NULL),
('Henry Taylor', 1, 110000, '2019-02-10', 1),
('Iris Chen', 5, 70000, '2021-01-10', NULL),
('Jack Wilson', 2, 78000, '2020-08-15', 4);

-- Insert products
INSERT INTO products (product_name, category, price, stock_quantity) VALUES
('Laptop Pro', 'Electronics', 1200, 50),
('Mouse Wireless', 'Electronics', 25, 200),
('Keyboard Mechanical', 'Electronics', 150, 75),
('Monitor 4K', 'Electronics', 400, 30),
('Office Chair', 'Furniture', 300, 20),
('Desk Adjustable', 'Furniture', 500, 15),
('Notebook Leather', 'Stationery', 15, 300),
('Pen Set Premium', 'Stationery', 35, 150);

-- Insert sales transactions
INSERT INTO sales (emp_id, product_id, sale_amount, sale_date, quantity) VALUES
(2, 1, 1200, '2024-01-05', 1),
(2, 2, 75, '2024-01-06', 3),
(5, 4, 1200, '2024-01-10', 3),
(5, 3, 300, '2024-01-12', 2),
(10, 5, 600, '2024-01-15', 2),
(10, 6, 500, '2024-01-18', 1),
(2, 7, 30, '2024-01-20', 2),
(5, 1, 2400, '2024-01-22', 2),
(10, 8, 70, '2024-01-25', 2),
(2, 4, 400, '2024-02-01', 1);
```

---
# SECTION 2: DATA MOVEMENT - IMPORT/EXPORT & BULK OPERATIONS

## The 'Why'
In data engineering, you frequently need to:
- **Import large CSV/text files** into database (LOAD DATA INFILE)
- **Handle duplicate records gracefully** without failing the entire batch (INSERT IGNORE)
- **Replace existing data** when the same record appears again (REPLACE)
- **Clear tables efficiently** without logging individual DELETE operations (TRUNCATE)

## The 'How'
Understanding these operations is crucial because they have different performance characteristics, logging implications, and rollback behaviors.

### 2.1: LOAD DATA INFILE - Bulk CSV Import

**Concept:** Fastest way to load data from external files directly into the database.

**Syntax:**
```sql
LOAD DATA INFILE '/path/to/file.csv'
INTO TABLE table_name
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 ROWS
(column1, column2, column3);
```

**Key Parameters:**
- `INFILE '/path'` - File must be readable by MySQL process (or use LOCAL)
- `FIELDS TERMINATED BY` - Delimiter (usually comma)
- `ENCLOSED BY` - Quote character (for handling embedded delimiters)
- `IGNORE 1 ROWS` - Skip header row

**Performance:** 10-100x faster than INSERT statements

**Example with Error Handling:**
```sql
LOAD DATA LOCAL INFILE '/tmp/employees.csv'
INTO TABLE employees
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 ROWS
(emp_name, dept_id, salary, hire_date)
SET manager_id = NULL;  -- Default value
```

**⚠️ Gotchas:**
- File must be on database server (or use `LOAD DATA LOCAL` - requires `--local-infile` flag)
- MySQL user needs `FILE` privilege (or `SUPER` privilege)
- Character encoding mismatches cause silent failures or garbled data
- NOT written to binary log in STATEMENT mode (replication issues!)
- Cannot use with WHERE clause or JOINs
- Locks table during load (blocks concurrent queries in MyISAM)
- If load fails halfway, partial data remains (must TRUNCATE and retry)

### 2.2: JSON Files - Handling Semi-Structured Data

**Concept:** Modern data pipelines often work with JSON files. You need to parse JSON, handle nested structures, and infer schemas.

**Key Scenarios:**
- API responses stored as JSON
- NoSQL-style data being imported to SQL database
- Nested/hierarchical data (arrays, objects within rows)

**Common Challenges:**
- Nested structures (arrays, objects) need flattening
- Inconsistent schemas (some rows missing fields)
- Large files need streaming to avoid memory issues
- Type inference errors (string vs number vs boolean)

**Python Libraries:**
- `json` - Basic JSON handling
- `pandas` - CSV/JSON with automatic schema inference
- `pyspark` - Distributed schema inference & SQL
- `pyarrow/parquet` - Columnar format with schema

---

### 2.3: Inferring Schema from Files

**Concept:** Automatically detect column names, types, and nullable constraints from file headers and data samples.

**Why It Matters:**
- Manual schema definition is error-prone
- Different file formats have different schema hints
- Avoids type conversion failures during import
- Critical for handling 3rd-party data feeds

**Methods:**
1. **Header-based** - Use first row as column names
2. **Sample-based** - Inspect first N rows to infer types
3. **Metadata-based** - Use embedded schema (Parquet, Avro)
4. **Statistical** - Analyze entire file for optimal types


In [ ]:
import pandas as pd
import json
import pyarrow.parquet as pq
from pyspark.sql import SparkSession
from typing import Dict, List, Tuple

# ============= JSON PARSING & SCHEMA INFERENCE =============

# Example 1: Basic JSON file reading with Pandas
json_data = '''
[
  {"id": 1, "name": "Alice", "salary": 50000, "hire_date": "2020-01-15"},
  {"id": 2, "name": "Bob", "salary": 60000, "hire_date": "2021-03-22"},
  {"id": 3, "name": "Charlie", "salary": null, "hire_date": "2022-06-10"}
]
'''

# Read and infer schema automatically
df_json = pd.read_json(pd.io.json.json_normalize(json.loads(json_data)), orient='index')
print("Inferred Schema from JSON:")
print(df_json.dtypes)
print("\nDataFrame:")
print(df_json)

# ============= NESTED JSON FLATTENING =============

# Example 2: Handle nested structures
nested_json = '''
[
  {
    "id": 1,
    "name": "Alice",
    "address": {"city": "NYC", "zip": "10001"},
    "skills": ["SQL", "Python"],
    "salary": 50000
  },
  {
    "id": 2,
    "name": "Bob",
    "address": {"city": "LA", "zip": "90001"},
    "skills": ["Java", "Spark"],
    "salary": 60000
  }
]
'''

data = json.loads(nested_json)
df_nested = pd.json_normalize(data)
print("\n\nFlattened Nested JSON:")
print(df_nested.dtypes)
print("\nFlattened Data:")
print(df_nested)

# ============= HEADER & SCHEMA INFERENCE =============

# Example 3: Infer schema from CSV header and sample data
csv_sample = """id,name,salary,hire_date,is_active
1,Alice,50000,2020-01-15,true
2,Bob,60000,2021-03-22,false
3,Charlie,55000,2022-06-10,true"""

df_csv = pd.read_csv(pd.io.common.StringIO(csv_sample))
print("\n\nSchema Inferred from CSV:")
print(df_csv.dtypes)
print("\nColumn Info:")
print(df_csv.info())

# ============= AUTOMATIC TYPE INFERENCE FUNCTION =============

def infer_schema_from_dataframe(df: pd.DataFrame) -> Dict[str, str]:
    """
    Infer SQL-compatible types from pandas DataFrame
    """
    type_mapping = {
        'object': 'VARCHAR(255)',
        'int64': 'BIGINT',
        'int32': 'INT',
        'float64': 'DOUBLE',
        'bool': 'BOOLEAN',
        'datetime64[ns]': 'DATETIME',
        'timedelta64[ns]': 'BIGINT'
    }
    
    schema = {}
    for col, dtype in df.dtypes.items():
        dtype_str = str(dtype)
        schema[col] = type_mapping.get(dtype_str, 'VARCHAR(255)')
    
    return schema

inferred_schema = infer_schema_from_dataframe(df_csv)
print("\n\nSQL Schema from DataFrame:")
for col, sql_type in inferred_schema.items():
    print(f"  {col} {sql_type},")

# ============= CLOUD FILE HANDLING (S3, GCS, AZURE) =============

# Example 4: Reading from cloud storage URIs (conceptual)
print("\n\nCloud File Reading Patterns:")
print("""
# AWS S3
df_s3 = pd.read_csv('s3://my-bucket/data/employees.csv')
df_s3_json = pd.read_json('s3://my-bucket/data/events.json', lines=True)

# Google Cloud Storage
df_gcs = pd.read_csv('gs://my-bucket/data/employees.csv')

# Azure Blob Storage
df_azure = pd.read_csv('az://container/data/employees.csv')

# With credentials
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
df = pd.read_csv(s3.open('s3://bucket/file.csv', 'rb'))
""")

# ============= PYSPARK SCHEMA INFERENCE =============

print("\n\nPySpark Schema Inference:")
print("""
spark = SparkSession.builder.appName("SchemaInference").getOrCreate()

# Auto-infer schema from JSON file (NDJSON - one JSON per line)
df_spark = spark.read.option("inferSchema", "true").json("path/to/data.json")
print(df_spark.schema)  # Prints StructType with all fields and types

# From CSV with header
df_csv_spark = spark.read.option("inferSchema", "true").option("header", "true").csv("path/to/file.csv")

# Manual schema definition (more control)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
schema = StructType([
    StructField("id", LongType(), False),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", StringType(), True)
])

df_manual = spark.read.schema(schema).json("path/to/data.json")
""")

### 2.4: Cloud File Operations - S3, GCS, Azure

**Concept:** Modern data engineering involves reading from cloud object storage. Different platforms require different authentication & libraries.

**Cloud Storage Platforms:**

| Platform | URI Scheme | Library | Notes |
|----------|-----------|---------|-------|
| AWS S3 | `s3://bucket/key` | `boto3`, `s3fs`, `pyarrow` | Most common, excellent Spark integration |
| Google Cloud | `gs://bucket/key` | `gcsfs`, `google-cloud-storage` | Built-in BigQuery integration |
| Azure Blob | `az://container/blob` | `adlfs`, `azure-storage-blob` | Works with Azure Synapse |
| MinIO (on-prem) | `s3://bucket/key` | `boto3` (configured for MinIO endpoint) | S3-compatible object storage |

**Key Challenges:**
- Authentication & credential management
- Handling large files (multi-part downloads)
- Network latency for remote files
- Cost optimization (direct queries vs full download)
- Schema validation across regions/buckets

**Best Practices:**
1. Use IAM roles instead of hardcoded credentials
2. Stream files instead of downloading full copies
3. Use columnar formats (Parquet) for cloud storage
4. Implement retry logic for network failures
5. Cache frequently accessed files locally

---

### 2.5: Type Inference Best Practices

**The Challenge:**
When reading arbitrary files, type inference can fail or guess wrong:
- `"123"` → Is this STRING or INTEGER?
- `"2024-01-15"` → Is this STRING or DATE?
- `null`, `None`, `""` → How to handle missing values?
- `1.0` vs `1` → Should this be FLOAT or INT?

**Strategies:**

1. **Sample-First Approach** (Recommended)
   - Read first 10,000 rows
   - Infer types from sample
   - Validate on full dataset

2. **Strict Type Definition**
   - Manually define schema upfront
   - Enforce via validation
   - Best for critical pipelines

3. **Statistical Approach**
   - Calculate type confidence scores
   - Use most likely type if >95% confidence
   - Flag ambiguous columns for review

4. **Format-Specific Metadata**
   - Parquet/ORC: Schema embedded in file
   - Avro: Schema in file header
   - CSV: No schema (most error-prone)
   - JSON: Infer from structure

**⚠️ Common Type Inference Errors:**
- Treating all decimals as FLOAT (precision loss)
- Missing encoding detection (UTF-8 vs Latin1)
- Treating booleans as STRING
- Not detecting date/timestamp formats consistently
- Assuming first row contains data (not always true)

In [ ]:
# ============= ADVANCED SCHEMA INFERENCE =============

from datetime import datetime, date
import re
from collections import Counter

def advanced_type_inference(value) -> str:
    """
    Advanced type inference for a single value.
    Returns: 'INT', 'FLOAT', 'BOOLEAN', 'DATE', 'DATETIME', 'VARCHAR'
    """
    if value is None or value == '' or str(value).lower() in ['null', 'none', 'na', 'nan']:
        return 'NULLABLE'
    
    str_val = str(value).strip()
    
    # Try boolean
    if str_val.lower() in ['true', 'false', 'yes', 'no', '0', '1']:
        return 'BOOLEAN'
    
    # Try integer
    try:
        int(str_val)
        return 'INT'
    except ValueError:
        pass
    
    # Try float
    try:
        float(str_val)
        return 'FLOAT'
    except ValueError:
        pass
    
    # Try date/datetime patterns
    date_patterns = [
        r'^\d{4}-\d{2}-\d{2}$',  # YYYY-MM-DD
        r'^\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}$',  # YYYY-MM-DD HH:MM:SS
        r'^\d{1,2}/\d{1,2}/\d{4}$',  # MM/DD/YYYY or DD/MM/YYYY
    ]
    
    if any(re.match(pattern, str_val) for pattern in date_patterns):
        if ' ' in str_val or 'T' in str_val:
            return 'DATETIME'
        else:
            return 'DATE'
    
    return 'VARCHAR'

def infer_column_type(values: List, confidence_threshold: float = 0.8) -> Tuple[str, float]:
    """
    Infer column type from sample values with confidence score.
    
    Args:
        values: List of values from a column
        confidence_threshold: Minimum % of values agreeing on type
    
    Returns:
        Tuple of (inferred_type, confidence_score)
    """
    if not values:
        return 'VARCHAR', 0.0
    
    # Filter out nulls
    non_null_values = [v for v in values if v is not None and str(v).strip() != '']
    
    if not non_null_values:
        return 'VARCHAR', 0.0  # All nulls
    
    # Count type votes
    type_votes = Counter()
    for val in non_null_values:
        inferred = advanced_type_inference(val)
        type_votes[inferred] += 1
    
    # Find most common type
    most_common_type, count = type_votes.most_common(1)[0]
    confidence = count / len(non_null_values)
    
    # Type precedence: more specific types win
    type_precedence = {'INT': 0, 'FLOAT': 1, 'DATE': 2, 'DATETIME': 3, 'BOOLEAN': 4, 'VARCHAR': 5}
    
    if confidence >= confidence_threshold:
        return most_common_type, confidence
    else:
        # Low confidence - default to VARCHAR
        return 'VARCHAR', confidence

# Example usage
sample_data = {
    'id': ['1', '2', '3', '4', '5'],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'salary': ['50000', '60000', '55000', None, '65000'],
    'hire_date': ['2020-01-15', '2021-03-22', '2022-06-10', '2023-01-05', '2024-02-14'],
    'is_active': ['true', 'false', 'true', 'true', 'false']
}

print("Advanced Type Inference Results:")
print("-" * 60)
for col, values in sample_data.items():
    inferred_type, confidence = infer_column_type(values)
    print(f"{col:15} → {inferred_type:10} (Confidence: {confidence:.1%})")

# ============= CLOUD FILE OPERATIONS (Boto3 Example) =============

print("\n\nCloud File Operations - AWS S3 Example:")
print("""
import boto3
import pandas as pd
from io import BytesIO

# Initialize S3 client
s3_client = boto3.client('s3')

# Read CSV from S3
def read_csv_from_s3(bucket: str, key: str) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(obj['Body'])
    return df

# Read JSON from S3 (NDJSON - newline delimited)
def read_ndjson_from_s3(bucket: str, key: str) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    df = pd.read_json(obj['Body'], lines=True)
    return df

# List all CSV files in bucket
def list_s3_files(bucket: str, prefix: str = '') -> list:
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    return [obj['Key'] for obj in response.get('Contents', [])]

# Usage
df = read_csv_from_s3('my-data-bucket', 'employees.csv')
inferred = infer_schema_from_dataframe(df)
""")

# ============= PANDAS DTYPE SPECIFICATION =============

print("\nPandas Dtype Specification for CSV Parsing:")
print("""
# Specify types explicitly to avoid inference errors
dtype_spec = {
    'id': 'int64',
    'name': 'string',
    'salary': 'Int64',  # Nullable integer
    'hire_date': 'datetime64[ns]',
    'is_active': 'bool'
}

df = pd.read_csv(
    'data.csv',
    dtype=dtype_spec,
    parse_dates=['hire_date'],
    na_values=['NULL', 'N/A', 'null']
)

# Convert after reading (alternative approach)
df['hire_date'] = pd.to_datetime(df['hire_date'])
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
""")

# ============= PARQUET FORMAT (Schema-Aware) =============

print("\nParquet Files - Built-in Schema:")
print("""
import pyarrow.parquet as pq

# Parquet files contain schema metadata
parquet_file = pq.read_table('data.parquet')
schema = parquet_file.schema

print(schema)  # Shows all columns, types, and nullability
# Output: 
# id: int64 (not null)
# name: string
# salary: int64 (nullable)
# hire_date: timestamp[ns][tz=UTC]

# Convert to pandas with schema preservation
df = parquet_file.to_pandas()

# Schema is already correct - no type inference needed!
print(df.dtypes)
""")

### 2.6: File Format Comparison & Best Practices

**Format Comparison Table:**

| Format | Compression | Schema | Speed | Use Case |
|--------|-------------|--------|-------|----------|
| **CSV** | Optional | None (inferred) | Slow | Human-readable, external APIs |
| **JSON** | Optional | None (inferred) | Medium | APIs, semi-structured data |
| **Parquet** | Built-in | Embedded | Very Fast | Data warehouses (Snowflake, BigQuery) |
| **ORC** | Built-in | Embedded | Very Fast | Hadoop/Hive ecosystem |
| **Avro** | Built-in | Embedded | Fast | Kafka, schema evolution |
| **Excel** | Native | None (inferred) | Slow | Business users, small datasets |

**Schema Inference Gotchas:**

❌ **Common Mistakes:**
1. **Not handling headers correctly**
   ```
   # Wrong - treats header as data
   df = pd.read_csv('file.csv', header=None)
   
   # Right
   df = pd.read_csv('file.csv', header=0)  # or header='infer'
   ```

2. **Assuming first row is consistent**
   ```
   # Some files have metadata rows before header
   # Solution: skip_rows parameter
   df = pd.read_csv('file.csv', skiprows=5)
   ```

3. **Type inference on small samples**
   ```
   # Reading 100 rows might infer wrong type
   df = pd.read_csv('huge_file.csv', nrows=100)
   
   # Better: read sample, infer, validate on full file
   sample = pd.read_csv('huge_file.csv', nrows=10000)
   inferred_types = infer_schema_from_dataframe(sample)
   df = pd.read_csv('huge_file.csv', dtype=inferred_types)
   ```

4. **Not handling missing values**
   ```
   # Different representations of NULL in wild data
   na_values = ['NULL', 'null', 'N/A', 'n/a', '', 'NA', 'None', '-']
   df = pd.read_csv('file.csv', na_values=na_values)
   ```

5. **Character encoding issues**
   ```
   # Wrong encoding → corrupted characters
   # Solution: specify encoding
   df = pd.read_csv('file.csv', encoding='utf-8')  # or 'latin1', 'cp1252'
   ```

**Recommended Workflow:**

```
1. INSPECTION
   ├─ Check file size
   ├─ Preview first few rows
   └─ Check encoding (file -i command)

2. SAMPLING
   ├─ Read 1-10% of file
   ├─ Infer schema from sample
   └─ Validate type confidence

3. VALIDATION
   ├─ Read full file with specified types
   ├─ Check for parsing errors
   ├─ Validate against business rules
   └─ Check row counts match expectations

4. TRANSFORMATION
   ├─ Flatten nested structures
   ├─ Handle missing values
   ├─ Type conversion if needed
   └─ Normalize column names

5. LOADING
   ├─ Generate CREATE TABLE statement
   ├─ Create table with inferred schema
   ├─ Insert data (or bulk load)
   └─ Verify row counts
```

**SQL Creation from Inferred Schema:**

```python
def generate_create_table_sql(
    dataframe: pd.DataFrame,
    table_name: str,
    primary_key: str = None
) -> str:
    """Generate CREATE TABLE statement from DataFrame schema"""
    
    columns = []
    for col, dtype in dataframe.dtypes.items():
        sql_type = {
            'object': 'VARCHAR(255)',
            'int64': 'BIGINT',
            'float64': 'DECIMAL(18,4)',
            'bool': 'BOOLEAN',
            'datetime64[ns]': 'DATETIME'
        }.get(str(dtype), 'VARCHAR(255)')
        
        nullable = 'NULL' if dataframe[col].isnull().any() else 'NOT NULL'
        columns.append(f"  {col} {sql_type} {nullable}")
    
    if primary_key:
        columns.append(f"  PRIMARY KEY ({primary_key})")
    
    return f"""CREATE TABLE {table_name} (
{','.join(columns)}
);"""
```

**When to Use Each Tool:**

| Scenario | Tool | Why |
|----------|------|-----|
| One-time CSV import | pandas | Simple, no dependencies |
| Large distributed data | PySpark | Handles petabytes, distributed |
| Nested/complex JSON | pandas + `json_normalize` | Easy flattening |
| Production pipeline | PySpark + schema manifest | Reproducible, no inference errors |
| Cloud data lake | Parquet files | Built-in schema, fast |
| Machine learning | DuckDB + Arrow | In-process, columnar |

---
# SECTION 3: MYSQL-NATIVE DATA HANDLING - JSON, CLOUD & SCHEMA INFERENCE

## Overview
While you can use external tools (Python, Spark) for data loading, MySQL has built-in capabilities for:
- **JSON file parsing** using `JSON_*` functions
- **Cloud file loading** via S3 plugins or external scripts
- **Schema detection** using information schema queries
- **Automatic type inference** through LOAD DATA analysis

This section covers **pure MySQL approaches** that don't require external tools.

---

### 3.1: Loading JSON Files in MySQL

**Concept:** MySQL 5.7+ has native JSON support. You can:
1. Load JSON as text, then parse it
2. Use `JSON_TABLE()` to extract and flatten nested structures
3. Import JSON line-by-line (NDJSON format)

**Method 1: LOAD DATA INFILE with JSON parsing**

```sql
-- Step 1: Load raw JSON into temporary table
CREATE TEMPORARY TABLE raw_json (
    json_data JSON
);

LOAD DATA INFILE '/path/to/data.json'
INTO TABLE raw_json
(json_data);

-- Step 2: Parse and extract using JSON_TABLE()
CREATE TABLE employees (
    id INT,
    name VARCHAR(100),
    salary INT,
    hire_date DATE
);

INSERT INTO employees
SELECT 
    JSON_EXTRACT(json_data, '$.id') as id,
    JSON_EXTRACT(json_data, '$.name') as name,
    JSON_EXTRACT(json_data, '$.salary') as salary,
    JSON_EXTRACT(json_data, '$.hire_date') as hire_date
FROM raw_json;
```

**Method 2: JSON_TABLE (MySQL 5.7.12+) - For nested structures**

```sql
-- Flatten nested JSON in a single query
INSERT INTO employees (id, name, salary, hire_date)
SELECT
    extracted.id,
    extracted.name,
    extracted.salary,
    extracted.hire_date
FROM (
    SELECT json_data FROM raw_json
) as source,
JSON_TABLE(
    source.json_data,
    '$[*]' COLUMNS (
        id INT PATH '$.id',
        name VARCHAR(100) PATH '$.name',
        salary INT PATH '$.salary',
        hire_date DATE PATH '$.hire_date'
    )
) AS extracted;
```

**Method 3: NDJSON (Newline-Delimited JSON)**

```sql
-- Each line is a separate JSON object - easier to parse
CREATE TEMPORARY TABLE raw_ndjson (
    json_line TEXT
);

LOAD DATA INFILE '/path/to/data.ndjson'
INTO TABLE raw_ndjson
LINES TERMINATED BY '\n'
(json_line);

-- Parse line by line
INSERT INTO employees
SELECT
    JSON_UNQUOTE(JSON_EXTRACT(json_line, '$.id')) as id,
    JSON_UNQUOTE(JSON_EXTRACT(json_line, '$.name')) as name,
    JSON_UNQUOTE(JSON_EXTRACT(json_line, '$.salary')) as salary,
    JSON_UNQUOTE(JSON_EXTRACT(json_line, '$.hire_date')) as hire_date
FROM raw_ndjson;
```

**⚠️ JSON Loading Gotchas:**
- File must be valid JSON (or NDJSON)
- Missing fields become NULL
- Type inference based on source (numbers stay INT, strings stay VARCHAR)
- Large files (>1GB) may cause memory issues
- No built-in validation for type consistency
- Character encoding issues common with international data

### 3.2: Loading from Cloud Storage (S3, GCS, Azure)

**Challenge:** MySQL cannot directly access cloud files. Solutions:

**Option 1: MySQL S3 Plugin (AWS)**

```sql
-- Install the S3 storage plugin
-- MySQL 8.0 only; requires AWS credentials

-- Prerequisites
-- 1. Configure AWS credentials in MySQL config
-- 2. Install s3_storage_engine plugin

-- Load from S3
LOAD DATA FROM S3 's3://mybucket/data/employees.csv'
INTO TABLE employees
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
IGNORE 1 ROWS;
```

**Option 2: MySQL 8.0 with aws_sdk library**

```sql
-- Use UDF (User Defined Function) to read from S3
-- Note: Requires compilation or pre-built binaries

-- Example with MariaDB (better S3 support)
LOAD DATA FROM S3 
  ENDPOINT_URL = 'https://s3.amazonaws.com'
  BUCKET = 'my-bucket'
  REGION = 'us-east-1'
  OBJECT_KEY = 'data/employees.csv'
INTO TABLE employees
FIELDS TERMINATED BY ','
IGNORE 1 ROWS;
```

**Option 3: Hybrid Approach (Recommended for pure MySQL)**

```bash
# Step 1: Download from cloud to local temp file
aws s3 cp s3://my-bucket/employees.csv /tmp/employees.csv

# Step 2: Load from local file
mysql -u user -p database << 'EOF'
LOAD DATA INFILE '/tmp/employees.csv'
INTO TABLE employees
FIELDS TERMINATED BY ','
IGNORE 1 ROWS;
EOF

# Step 3: Clean up
rm /tmp/employees.csv
```

**Option 4: Using MySQL with Python wrapper**

```python
import boto3
import subprocess

s3 = boto3.client('s3')

# Download from S3
s3.download_file(
    'my-bucket',
    'employees.csv',
    '/tmp/employees.csv'
)

# Load into MySQL
subprocess.run([
    'mysql', '-u', 'user', '-ppassword', 'database',
    '-e', 'LOAD DATA INFILE "/tmp/employees.csv" INTO TABLE employees'
])
```

**⚠️ Cloud Loading Gotchas:**
- AWS S3 plugin only works on MySQL 8.0+ (and is complex to set up)
- Most production systems use hybrid approach (download first)
- GCS has no native MySQL plugin
- Azure has limited support
- Need proper IAM credentials for security

### 3.3: Schema Inference in MySQL - Header Analysis

**Concept:** MySQL can analyze CSV headers and data to infer table structure automatically.

**Method 1: Parse CSV Header to Infer Columns**

```sql
-- Step 1: Load raw CSV into staging table (all as text)
CREATE TEMPORARY TABLE raw_data (
    raw_line TEXT
);

LOAD DATA INFILE '/path/to/file.csv'
INTO TABLE raw_data
LINES TERMINATED BY '\n'
(raw_line);

-- Step 2: Extract header (first row)
SELECT 
    SUBSTRING_INDEX(SUBSTRING_INDEX(raw_line, ',', numbers.n), ',', -1) AS column_name
FROM (
    SELECT 1 as n UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4 UNION ALL SELECT 5
) numbers
WHERE numbers.n <= (LENGTH(raw_line) - LENGTH(REPLACE(raw_line, ',', '')) + 1)
  AND raw_line = (SELECT raw_line FROM raw_data LIMIT 1);
```

**Method 2: Dynamic Column Extraction**

```sql
-- Store header information in a helper table
CREATE TABLE column_metadata (
    table_name VARCHAR(100),
    column_position INT,
    column_name VARCHAR(100),
    inferred_type VARCHAR(50),
    nullable BOOLEAN,
    max_length INT
);

-- Populate after analyzing data
INSERT INTO column_metadata
SELECT 
    'employees' as table_name,
    1 as column_position,
    'id' as column_name,
    'INT' as inferred_type,
    FALSE as nullable,
    NULL as max_length
UNION ALL SELECT 'employees', 2, 'name', 'VARCHAR', TRUE, 100
UNION ALL SELECT 'employees', 3, 'salary', 'DECIMAL', TRUE, NULL;
```

**Method 3: Inferring Column Count and Names**

```sql
-- Function to count columns in CSV line
DELIMITER $$

CREATE FUNCTION count_csv_fields(line TEXT, delimiter CHAR(1)) 
RETURNS INT
DETERMINISTIC
READS SQL DATA
BEGIN
    DECLARE field_count INT DEFAULT 1;
    IF line IS NULL OR line = '' THEN
        RETURN 0;
    END IF;
    SET field_count = 1 + (LENGTH(line) - LENGTH(REPLACE(line, delimiter, '')));
    RETURN field_count;
END$$

DELIMITER ;

-- Usage
SELECT count_csv_fields('id,name,salary,hire_date', ',') AS num_columns;
-- Result: 4
```

**Method 4: Extracting Specific Column Values**

```sql
DELIMITER $$

CREATE FUNCTION get_csv_field(
    line TEXT,
    field_position INT,
    delimiter CHAR(1)
) RETURNS VARCHAR(1000)
DETERMINISTIC
READS SQL DATA
BEGIN
    DECLARE result VARCHAR(1000);
    DECLARE idx INT;
    DECLARE field_count INT;
    
    SET field_count = 1 + (LENGTH(line) - LENGTH(REPLACE(line, delimiter, '')));
    
    IF field_position > field_count THEN
        RETURN NULL;
    END IF;
    
    RETURN TRIM(SUBSTRING_INDEX(SUBSTRING_INDEX(line, delimiter, field_position), delimiter, -1));
END$$

DELIMITER ;

-- Usage to extract header
SELECT
    get_csv_field((SELECT raw_line FROM raw_data LIMIT 1), 1, ',') AS col1,
    get_csv_field((SELECT raw_line FROM raw_data LIMIT 1), 2, ',') AS col2,
    get_csv_field((SELECT raw_line FROM raw_data LIMIT 1), 3, ',') AS col3;
```

### 3.4: Type Inference in MySQL

**Concept:** Analyze sample data to automatically detect column types (INT, VARCHAR, DATE, etc.)

**Method 1: Simple Type Detection Function**

```sql
DELIMITER $$

CREATE FUNCTION infer_column_type(value VARCHAR(1000))
RETURNS VARCHAR(50)
DETERMINISTIC
READS SQL DATA
BEGIN
    DECLARE result VARCHAR(50);
    
    -- Check for NULL/empty
    IF value IS NULL OR value = '' THEN
        RETURN 'VARCHAR';
    END IF;
    
    -- Check for boolean
    IF value IN ('true', 'false', 'yes', 'no', '0', '1', 'True', 'False') THEN
        RETURN 'BOOLEAN';
    END IF;
    
    -- Check for integer
    IF value REGEXP '^-?[0-9]+$' THEN
        RETURN 'INT';
    END IF;
    
    -- Check for decimal/float
    IF value REGEXP '^-?[0-9]+\\.[0-9]+$' THEN
        RETURN 'DECIMAL(18,4)';
    END IF;
    
    -- Check for date (YYYY-MM-DD)
    IF value REGEXP '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' THEN
        RETURN 'DATE';
    END IF;
    
    -- Check for datetime (YYYY-MM-DD HH:MM:SS)
    IF value REGEXP '^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$' THEN
        RETURN 'DATETIME';
    END IF;
    
    -- Default to VARCHAR
    RETURN 'VARCHAR';
END$$

DELIMITER ;

-- Usage
SELECT 
    infer_column_type('123') AS type1,           -- INT
    infer_column_type('45.67') AS type2,         -- DECIMAL
    infer_column_type('2024-01-15') AS type3,    -- DATE
    infer_column_type('Alice') AS type4;         -- VARCHAR
```

**Method 2: Multi-Value Type Consensus**

```sql
-- Analyze multiple values from a column to determine best type
CREATE PROCEDURE infer_column_type_consensus(
    IN p_table_name VARCHAR(100),
    IN p_column_name VARCHAR(100),
    IN p_sample_size INT,
    OUT p_inferred_type VARCHAR(50),
    OUT p_confidence DECIMAL(3,2)
)
BEGIN
    DECLARE v_int_count INT DEFAULT 0;
    DECLARE v_decimal_count INT DEFAULT 0;
    DECLARE v_date_count INT DEFAULT 0;
    DECLARE v_total_count INT DEFAULT 0;
    DECLARE v_max_length INT DEFAULT 0;
    
    SET @sql := CONCAT(
        'SELECT COUNT(*) INTO @total FROM (SELECT ', p_column_name, 
        ' FROM ', p_table_name, ' LIMIT ', p_sample_size, ') AS sample'
    );
    PREPARE stmt FROM @sql;
    EXECUTE stmt;
    SET v_total_count = @total;
    
    -- Count values that look like integers
    SET @sql := CONCAT(
        'SELECT COUNT(*) INTO @int_cnt FROM (SELECT ', p_column_name,
        ' FROM ', p_table_name, 
        ' WHERE ', p_column_name, ' REGEXP "^-?[0-9]+$" LIMIT ', p_sample_size, 
        ') AS sample'
    );
    PREPARE stmt FROM @sql;
    EXECUTE stmt;
    SET v_int_count = @int_cnt;
    
    -- Count values that look like dates
    SET @sql := CONCAT(
        'SELECT COUNT(*) INTO @date_cnt FROM (SELECT ', p_column_name,
        ' FROM ', p_table_name,
        ' WHERE ', p_column_name, ' REGEXP "^[0-9]{4}-[0-9]{2}-[0-9]{2}$" LIMIT ', p_sample_size,
        ') AS sample'
    );
    PREPARE stmt FROM @sql;
    EXECUTE stmt;
    SET v_date_count = @date_cnt;
    
    -- Find max string length
    SET @sql := CONCAT(
        'SELECT MAX(CHAR_LENGTH(', p_column_name, ')) INTO @max_len FROM ',
        p_table_name, ' LIMIT ', p_sample_size
    );
    PREPARE stmt FROM @sql;
    EXECUTE stmt;
    SET v_max_length = COALESCE(@max_len, 100);
    
    -- Determine type based on confidence threshold
    IF v_int_count / v_total_count >= 0.95 THEN
        SET p_inferred_type = 'INT';
        SET p_confidence = v_int_count / v_total_count;
    ELSEIF v_date_count / v_total_count >= 0.80 THEN
        SET p_inferred_type = 'DATE';
        SET p_confidence = v_date_count / v_total_count;
    ELSE
        SET p_inferred_type = CONCAT('VARCHAR(', v_max_length + 50, ')');
        SET p_confidence = 1.00;
    END IF;
    
    DEALLOCATE PREPARE stmt;
END$$

-- Usage
CALL infer_column_type_consensus('employees', 'id', 1000, @type, @conf);
SELECT @type AS inferred_type, @conf AS confidence;
```

**Method 3: Analyze Entire Table Schema**

```sql
-- Generate CREATE TABLE statement from existing data
CREATE PROCEDURE generate_schema_from_data(
    IN p_table_name VARCHAR(100),
    IN p_sample_size INT
)
BEGIN
    -- Create temp table to store schema info
    CREATE TEMPORARY TABLE temp_schema (
        column_name VARCHAR(100),
        inferred_type VARCHAR(100),
        confidence DECIMAL(3,2)
    );
    
    -- For each column, determine type
    -- (This would iterate through INFORMATION_SCHEMA in production)
    
    -- Output CREATE TABLE statement
    SELECT CONCAT(
        'CREATE TABLE ', p_table_name, '_new (\n',
        GROUP_CONCAT(
            CONCAT('  ', column_name, ' ', inferred_type)
            ORDER BY column_name
            SEPARATOR ',\n'
        ),
        '\n);'
    ) AS create_table_statement
    FROM temp_schema;
END$$
```